# C10 事前登記 02：競賽二（棋力預測）逐盤輸出與接著訓練（Colab 版）

**用途**：`docs/PREREG_02_OPENING_FUSION.md` 第 1.1 節的 Colab 部分，一次執行做完：

1. **只讀 prereg01**：讀 prereg01 的權重、`state.pt`、機率檔、分片備份。任何寫進 prereg01 目錄的動作都會被擋下，開頭和結尾比對 prereg01 目錄的檔案指紋。
2. **存逐盤輸出**：prereg01 epoch 1–4（原方向）、epoch 5（8 種對稱）在 inner（73,556 盤）和本機驗證題（19,156 盤）的 logits。
3. **重現核對**：三項都過才繼續，任一項不過就停下，不訓練。
4. **接著訓練**：從 prereg01 的 `state.pt` 接 4 個 epoch（編號 6–9），學習率從 1e-3 線性降到 0。指定的 checkpoint 是 `epoch_9.pth`，不另選 best。
5. **存 epoch 9 的逐盤輸出**：8 種對稱。

合併、校準、開局特徵、打分都在本機做，不在這裡。這份 notebook 不讀任何答案檔（打包檔裡本來就沒有）。

## 怎麼跑

0. 用**跑 prereg01 的同一個 Google 帳號**：prereg01 的權重和 `state.pt` 在它的 Drive（`MyDrive/C10/results/prereg01/`）。Drive 要剩 3 GB 以上。
1. 把 `prereg02_rank.ipynb` 放進 `MyDrive/C10/`。打包檔 `C10_bundle_v1.zip` 不用重傳。
2. 選單「執行階段 → 變更執行階段類型」：選 **G4 GPU**。G4 是灰色、點不了時，選 **A100 GPU**，「大量 RAM」關掉。不要選「CPU」：沒有 GPU 時，前面幾格就會停下。Colab 改派別的 GPU（不是 G4 或 A100）時也會停下。
3. 選單「執行階段 → 全部執行」。第一個程式格會要求存取 Google Drive，按允許。
4. 跑完後（G4 約 45 分鐘，推估），最後兩行是「Drive 已寫入完成並卸載」和「完成。請下載…」。這時 zip 已經在 Drive 上：先按「執行階段 → 中斷連線並刪除執行階段」（開著會一直扣運算單元），再下載 `MyDrive/C10/results/prereg02/rank_results.zip`，交給 Claude 的方式照 `docs/COLAB_RUN.md`。

詳細步驟以 `docs/COLAB_RUN.md` 的「事前登記 02」一節為準；和這裡不一致時，以那一節為準。

## 會停下來的情況（錯誤訊息裡都寫了處理方式）

- **開頭的檢查（preflight）**：找不到 prereg01 的檔、prereg01 的檔和本機記錄的不同、`best_model.pth` 不等於 `epoch_5.pth`、Drive 空間不足、prereg01 目錄和第一次執行時不同。這時還沒開始算，但執行階段開著就會一直扣運算單元：截圖後先按「執行階段 → 中斷連線並刪除執行階段」。
- **沒有 GPU，或拿到的 GPU 不是 G4／A100**：在設定 `DEVICE` 的那一格停下，還沒開始算。照訊息先刪除執行階段，換成 G4（灰色時 A100）再全部執行。
- **重現核對沒過**：不訓練。這時會先打包 `MyDrive/C10/results/prereg02/rank_results_repro_failed.zip`（含逐盤輸出，查原因要用），再停下。下載這個 zip，連同錯誤訊息的截圖給 Claude（寫入已經送出，可以先刪除執行階段）。
- **每個 epoch 的步數和 prereg01 不同**（分片是重抽的才可能發生）：在接著訓練前停下，逐盤輸出已存好。截圖給 Claude。
- **prereg02 的 `ckpt/state.pt` 讀不進來或設定不同**：照錯誤訊息把 `prereg02/rank/ckpt` 改名保留，再全部執行。
- **結尾發現 prereg01 目錄被改動**：結果照樣打包，但會停下來；截圖給 Claude。

## 結果寫到哪裡

`MyDrive/C10/results/prereg02/rank/`：

| 檔案 | 內容 |
|---|---|
| `dumps/inner_keys.npz` | inner 每盤一列：`rank`、`game_id`、`player_id`、`color`、`label`（0–9，D…6D）。所有 `inner_*.npz` 的列順序都和它相同 |
| `dumps/inner_<名稱>.npz` | `logits` float32 [盤數, 對稱數, 10]、`weights_sha256`、`syms`。名稱：`p01e1`–`p01e4`（原方向）、`p01e5_d4`、`p02e9_d4`（8 種對稱） |
| `dumps/val_keys.npz` | 本機驗證題每個 slot 一列：`question_id`、`slot`（從 1 起算）、`color`、`parsed`；題目檔的列順序，再依 slot 順序 |
| `dumps/val_<名稱>.npz` | 同上的 logits；解析失敗的 slot 整列是 NaN |
| `dumps/_DONE_<檔名>.json` | 每個 dump 的完成標記：列數、SHA-256、權重 SHA-256、秒數，以及算它的那次執行的 GPU 型號、torch、cuDNN 版本 |
| `repro_check.json` | 三項重現核對的數字和結果（② 的容許差依「算 `val_p01e5_d4` 那次執行的 GPU」決定） |
| `ckpt/history.csv` | epoch 6–9 的訓練紀錄和 inner 逐盤 accuracy（只當監看） |
| `ckpt/epoch_6..9.pth`、`ckpt/state.pt` | 權重和接續用的狀態（不打包；指定的 checkpoint 是 `epoch_9.pth`） |
| `p01_readonly_before.json`、`p01_readonly_after.json` | prereg01 目錄每個檔的大小和 mtime（開頭、結尾） |
| `config.json`、`env.json`、`run_log.json`、`timings.json`、`leak_check.json` | 設定、環境、每次執行的紀錄、各階段秒數、`load_rows` 的外洩檢查 |

`MyDrive/C10/results/prereg02/rank_results.zip`：上面除了權重以外的全部檔案，這個檔帶回本機。

## 中斷後怎麼接續

再按一次「全部執行」就好：

- 已完成的逐盤輸出靠完成標記和 SHA-256 跳過。
- 分片特徵依序從 prereg02、prereg01 的 Drive 備份還原；都不能用才重抽（約多 6 分鐘），抽完備份到 prereg02。
- 接著訓練從 prereg02 的 `ckpt/state.pt` 接續。斷在 epoch 中間時，那個 epoch 從頭重跑，學習率排程從已存的步數接續。
- 資料順序和 prereg01 一樣不固定（webdataset 的洗牌），接續過的執行無法逐位元重現；每個 epoch 開始時 `random`、`numpy`、`torch` 的 seed 設為 42＋epoch。
- 換了 GPU 接續時，已存的逐盤輸出不重算；重現核對 ② 的容許差看的是當初算它的那次執行的 GPU（記在完成標記），不會因為這次換了 GPU 而改變。

## 接著訓練做不完時（收尾開關）

**只有 Claude 請你這樣做時才用**（例如運算單元不夠、一直斷線，第 9 個 epoch 跑不完）：

1. 在第一個程式格（設定格）開頭附近找到 `FINISH_WITHOUT_TRAINING = False`，改成 `FINISH_WITHOUT_TRAINING = True`。
2. 選單「執行階段 → 全部執行」。

這樣不會再接著訓練：已存的逐盤輸出照樣核對，第 9 個 epoch 沒完成就不做 epoch 9 的逐盤輸出，結尾照樣打包 `rank_results.zip`。本機照登記（PREREG_02 1.2）拿掉用到 epoch 9 的組合。prereg01 的逐盤輸出都已存好時，執行階段可以選 CPU；有缺的話會停下，請你換回 GPU。

## 和 prereg01 notebook 的差異

底稿是已 commit 的 `colab/prereg01_rank.ipynb`（產生時核對 SHA-256）。沿用的格子沒列在下表的，和 prereg01 逐字相同；和官方 notebook 的差異照 prereg01 的差異表（`config.json` 的 `deviations_from_official`）。逐格的 unified diff 在 `results/prereg02/notebook_diff_rank.txt`。

| 格 | 改了什麼 |
|---|---|
| 新增 | 第一格 markdown：用途、步驟、會停下的情況、輸出清單、接續方式、這張差異表。 |
| 設定 | 結果目錄改成 `prereg02`（SMOKE 是 `prereg02_smoke`），分片備份 `FEAT_BACKUP` 跟著改到 `prereg02/features_rank`。新增只讀的 `P01_ROOT`／`P01_RANK`／`P01_CKPT`／`P01_FEAT`（`prereg01`，SMOKE 是 `prereg01_smoke`）、`guard_write()`（目的地落在 `P01_ROOT` 底下就停）、`p01_fingerprint()`（檔名、大小、mtime）。`write_json` 先過 `guard_write`。`SMOKE_CFG` 不動（分片備份的指紋含它，改了就還原不到 prereg01 的分片），本版的 SMOKE 設定另放 `SMOKE_CFG_P02`。`config.json` 的 `prereg` 改指 PREREG_02，另記底稿 notebook 的 SHA-256 和這張表。新增收尾開關 `FINISH_WITHOUT_TRAINING = False`（登記 1.2「第 9 個 epoch 沒跑完」的退路；只在 Claude 請你改時改成 True）。 |
| 新增 preflight | 開跑前確認 prereg01 的 `epoch_1..5.pth`、`best_model.pth`、`state.pt`、`history.csv`、`rank_val_probs.csv`、`env.json` 都在，記 SHA-256；`best_model.pth` 必須等於 `epoch_5.pth`；正式執行時機率檔、`history.csv`、`env.json` 必須和本機 commit 的 prereg01 結果逐位元組相同，inner accuracy 目標是 25.1482%、每個 epoch 2,814 步。prereg01 沒有分片備份只警告（會重抽）。Drive 剩餘空間 ≥ 3 GB。記下 prereg01 目錄的指紋 `p01_readonly_before.json`；重跑時和第一次記的不同就停。 |
| 底稿官方 2 | `NUM_EPOCHS` 維持 prereg01 的 5（SMOKE 是 prereg01 冒煙的 1），用來比對 `state.pt`。新增 `EXTRA_EPOCHS`＝4（SMOKE 是 1）、`PEAK_LR`＝`LEARNING_RATE`＝1e-3、`FINAL_EPOCH`＝`NUM_EPOCHS`＋`EXTRA_EPOCHS`。`CKPT_DIR` 由設定格決定（`prereg02/rank/ckpt`），`makedirs` 前先過 `guard_write`。最後的 `print` 多印 GPU 型號和接著訓練的 epoch 數。正式執行沒有 GPU 就在這裡停下（收尾開關打開時例外，缺的逐盤輸出另由 dump 工具擋下）；有 GPU 但型號（環境紀錄格 nvidia-smi 查到的）不是 G4（RTX PRO 6000）或 A100 也停下（PREREG_02 第 7 節第 8 條）。 |
| 底稿分片工具 | `restore_features(fp, src)` 改成可以指定來源，對來源只讀；還原到本機前 `guard_write(FEAT_DIR)`。`backup_features` 只寫到 `prereg02/features_rank`，`rmtree` 和每個複製目的地先過 `guard_write`。 |
| 底稿官方 9 | 依序試 `prereg02/features_rank`、`prereg01/features_rank` 還原分片；都不能用才重抽（抽完備份到 prereg02）。`rmtree(FEAT_DIR)` 前先過 `guard_write`。`run_log.json` 另記分片來源。 |
| 新增 dump 工具 | `d4(x, s)`（s＝0 原方向；s≥4 先在 dim 3 鏡射，再在 dims (2, 3) 旋轉 s%4 次，格內自我檢查 8 種兩兩不同且封閉）；`load_net()`（讀進記憶體、算 SHA-256、從同一份位元組載入）；inner 用分片的 `__key__`＝`等級_game_id` 對回 `df_val` 的列，assert 每盤剛好一次；本機驗證題照題目檔列順序和 slot 1..num_games 解析，失敗的保留位置（logits 填 NaN），每題一個 batch（和 prereg01 推論相同）；每個檔先寫 `.tmp` 再改名，附 `_DONE_<檔名>.json`（另記算它的那次執行的 GPU 型號、torch、cuDNN），重跑時對得上就跳過。正式執行時要算 dump 卻沒有 GPU 就停。另有 `record_p01_after()`（結尾指紋）、`pack_results()`（打包，跳過權重、暫存檔和 ≥ 100 MB 的檔）、`flush_drive()`（Colab 上 `drive.flush_and_unmount()`）。 |
| 新增 P1 輸出與重現核對 | prereg01 epoch 1–4 原方向、epoch 5 八種對稱的逐盤 logits（inner、本機驗證題）和兩個 key 檔。重現核對三項：① epoch 5 在 inner 的逐盤 accuracy 和 `history.csv` 相差 ≤ 0.05 個百分點；② 每題 float32 平均機率和 prereg01 `rank_val_probs.csv` 最大差距 ≤ 1e-4（算 `val_p01e5_d4` 那次執行的 GPU 型號和 prereg01 不同時 1e-3；正式執行沒記到 GPU 就不通過）；③ 規則 B 不同的題 ≤ 4（SMOKE 依題數等比例縮小）。任一項不過就停，不訓練；停下前記結尾指紋、打包 `prereg02/rank_results_repro_failed.zip`、送出 Drive 的寫入。 |
| 新增接著訓練 | 從 prereg01 `state.pt`（只讀、核對 SHA-256、設定、逐張量等於 `epoch_5.pth`）載入 model 和 AdamW 狀態，接著訓練 epoch 6–9：`LambdaLR` 每步 `max(0, 1 − t/T)`、T＝4×2,814，每個 epoch 開始 seed＝42＋epoch，其餘和 prereg01 相同。每個 epoch 存 `epoch_N.pth`、`state.pt`（含 scheduler、global_step、history、設定）、`history.csv`（inner 驗證只當監看）。有 prereg02 的 `state.pt` 就接續，設定不同就停。不產生 `best_model.pth`。訓練前用 `_count_w*.json` 和分片清單算出每個 epoch 的步數（webdataset 先依 worker 分分片），和 prereg01 的 2,814 步不同就停。收尾開關打開時不訓練，已完成的 epoch 從 prereg02 的 `state.pt` 讀回。 |
| 新增 P2 輸出 | 用 `epoch_9.pth` 存 8 種對稱的逐盤 logits（inner、本機驗證題）；第 9 個 epoch 沒完成（包括收尾開關打開時）就跳過並記錄。 |
| 底稿結尾（改寫） | 改成核對所有 dump 的列數、形狀、NaN 位置和完成標記；記 `p01_readonly_after.json` 並和開頭比對（不同就在打包後停下）；補完 config／timings／run_log；打包 `prereg02/rank_results.zip`（跳過 `.pth`、`.pt`、`.tmp` 和 ≥ 100 MB 的檔；dump 一律 < 100 MB，否則在寫檔時就停）；最後 `drive.flush_and_unmount()`。 |
| 拿掉 | prereg01 的設定紀錄與接續工具、官方 16（訓練迴圈）、18（訓練曲線）、20、21（推論）、所有官方 markdown 格。推論改由 dump 工具做，訓練改由接著訓練格做。 |
| metadata | notebook metadata 的 Colab 設定拿掉預選的 A100 和「大量 RAM」（照第一格選 G4；拿不到 G4 時選 A100 標準版）。每格 cell id 是 `c10-rank02-NN`，cell metadata `c10` 記底稿的格號、底稿的 `c10` 和是否改動。 |

## 其他

- **本機冒煙測試**：`colab/run_local_smoke.py` 設定 `C10_SMOKE=1`，只用極少量資料在 CPU 上跑。先在同一個 `--scratch` 跑 prereg01 的冒煙測試（產生 `prereg01_smoke`），再跑這份；結果寫到 `results/prereg02_smoke/`。SMOKE 時 prereg01 只有 1 個 epoch，`p01e1`–`p01e5_d4` 都用它，接著訓練 1 個 epoch，`p02e9_d4` 用那個 epoch。
- **不在這裡打分**：答案檔不上傳 Colab。

In [ ]:
# 設定：判斷在 Colab 還是本機冒煙測試，決定路徑和結果目錄。Colab 和本機只有這一格的寫法不同。
import csv
import datetime
import json
import multiprocessing
import os
import shutil
import sys
import time

RUN_START = time.time()

# 本版（prereg02）收尾開關：平常維持 False。只有 Claude 請你改的時候才改成 True（見第一格「接著訓練做不完時」）。
# True：不再接著訓練；已存的逐盤輸出照樣核對，第 9 個 epoch 沒完成就照登記不做 epoch 9 的逐盤輸出，結尾照樣打包。
FINISH_WITHOUT_TRAINING = False


def now_iso():
    return datetime.datetime.now().astimezone().isoformat(timespec='seconds')


IN_COLAB = 'google.colab' in sys.modules
SMOKE = os.environ.get('C10_SMOKE') == '1'   # Colab 上沒有這個環境變數，所以永遠是 False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/C10'
    WORK = '/content/c10'
else:
    DRIVE_DIR = os.environ['C10_DRIVE_DIR']
    WORK = os.environ['C10_WORK_DIR']
    if os.name == 'nt' and not SMOKE:
        # 官方的多進程依賴 Linux 的 fork；原生 Windows 只能跑冒煙測試（測試器會把進程換成執行緒）
        raise RuntimeError('原生 Windows 不支援正式執行，請在 Colab 上跑')

# 官方的多進程（抽特徵的 Process、推論的 Pool、DataLoader 的 worker）用的是 notebook 內定義的函式，只有 fork 能傳給子進程。
# Python 3.14 起 Linux 預設改成 forkserver，所以明確指定 fork；3.13 以前本來就是 fork，行為不變。
if sys.platform.startswith('linux'):
    multiprocessing.set_start_method('fork', force=True)
MP_START_METHOD = multiprocessing.get_start_method()

# 官方 notebook 用相對路徑（./training、./rank-features），這裡一律在 WORK 底下執行。
# webdataset 會把 Windows 路徑的「C:」當成網址格式，所以本機冒煙測試也必須用相對路徑。
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)

# 結果目錄：正式和 SMOKE 分開，絕不混用。本版（prereg02）只寫 prereg02；prereg01 只讀
RESULT_ROOT = os.path.join(DRIVE_DIR, 'results', 'prereg02_smoke' if SMOKE else 'prereg02')
RESULT_DIR = os.path.join(RESULT_ROOT, 'rank')
RESULT_CKPT = os.path.join(RESULT_DIR, 'ckpt')
FEAT_BACKUP = os.path.join(RESULT_ROOT, 'features_rank')   # 本版自己的分片備份（只在重抽時寫）

# 本版：prereg01 的結果目錄（權重、state.pt、機率檔、分片備份）只讀
P01_ROOT = os.path.join(DRIVE_DIR, 'results', 'prereg01_smoke' if SMOKE else 'prereg01')
P01_RANK = os.path.join(P01_ROOT, 'rank')
P01_CKPT = os.path.join(P01_RANK, 'ckpt')
P01_FEAT = os.path.join(P01_ROOT, 'features_rank')


def _real(path):
    return os.path.normcase(os.path.realpath(path))


def _is_under(path, root):
    p, r = _real(path), _real(root)
    return p == r or p.startswith(r.rstrip(os.sep) + os.sep)


def guard_write(path):
    # 本版：寫檔、torch.save、np.savez、copyfile 的目的地、rmtree、zip 一律先過這一關；落在 prereg01 底下就停
    if _is_under(path, P01_ROOT):
        raise PermissionError('本版不能寫入 prereg01 的目錄（只讀）：%s' % path)
    return path


def p01_fingerprint():
    # prereg01 目錄每個檔的 {相對路徑: [bytes, mtime_ns]}；開頭和結尾各記一次，結尾必須相同
    out = {}
    for root, dirs, files in os.walk(P01_ROOT):
        dirs.sort()
        for n in sorted(files):
            p = os.path.join(root, n)
            st = os.stat(p)
            out[os.path.relpath(p, P01_ROOT).replace(os.sep, '/')] = [st.st_size, st.st_mtime_ns]
    return out


# 本機工作目錄、prereg02 的結果目錄都不能和 prereg01 互相包含
# （沿用的準備格、抽特徵的 worker 只寫 WORK 底下，所以不另外過 guard_write）
for _p in (WORK, RESULT_ROOT):
    guard_write(_p)
    assert not _is_under(P01_ROOT, _p), (P01_ROOT, _p)
guard_write(RESULT_CKPT)
os.makedirs(RESULT_CKPT, exist_ok=True)

# SMOKE 的縮小方式：只在本機冒煙測試生效，正式執行時完全不用
SMOKE_CFG = {
    'fit_games': 2000,     # load_rows 全部通過之後，才用固定 seed 抽
    'inner_games': 400,
    'sample_seed': 42,
    'nproc': 2,            # 抽特徵和推論的進程數
    'per_shard': 300,      # 讓每個 worker 寫出好幾個分片，測到換檔
    'batch_size': 32,
    'num_epochs': 1,
    'questions': 12,       # 推論只做前 12 題
    'loader_workers': 0,   # 本機 Windows 的 spawn 無法 pickle notebook 內定義的函式
}

# 本版（prereg02）的 SMOKE 設定另外放；SMOKE_CFG 不動（分片備份的指紋含它，改了就還原不到 prereg01 的分片）
SMOKE_CFG_P02 = {
    'extra_epochs': 1,     # 接著訓練的 epoch 數（正式是 4）
}

# 本版和 prereg01 notebook 的差異（與第一格的表同一份）
P02_CHANGES = [
    "[新增] 第一格 markdown：用途、步驟、會停下的情況、輸出清單、接續方式、這張差異表。",
    "[設定] 結果目錄改成 `prereg02`（SMOKE 是 `prereg02_smoke`），分片備份 `FEAT_BACKUP` 跟著改到 `prereg02/features_rank`。新增只讀的 `P01_ROOT`／`P01_RANK`／`P01_CKPT`／`P01_FEAT`（`prereg01`，SMOKE 是 `prereg01_smoke`）、`guard_write()`（目的地落在 `P01_ROOT` 底下就停）、`p01_fingerprint()`（檔名、大小、mtime）。`write_json` 先過 `guard_write`。`SMOKE_CFG` 不動（分片備份的指紋含它，改了就還原不到 prereg01 的分片），本版的 SMOKE 設定另放 `SMOKE_CFG_P02`。`config.json` 的 `prereg` 改指 PREREG_02，另記底稿 notebook 的 SHA-256 和這張表。新增收尾開關 `FINISH_WITHOUT_TRAINING = False`（登記 1.2「第 9 個 epoch 沒跑完」的退路；只在 Claude 請你改時改成 True）。",
    "[新增 preflight] 開跑前確認 prereg01 的 `epoch_1..5.pth`、`best_model.pth`、`state.pt`、`history.csv`、`rank_val_probs.csv`、`env.json` 都在，記 SHA-256；`best_model.pth` 必須等於 `epoch_5.pth`；正式執行時機率檔、`history.csv`、`env.json` 必須和本機 commit 的 prereg01 結果逐位元組相同，inner accuracy 目標是 25.1482%、每個 epoch 2,814 步。prereg01 沒有分片備份只警告（會重抽）。Drive 剩餘空間 ≥ 3 GB。記下 prereg01 目錄的指紋 `p01_readonly_before.json`；重跑時和第一次記的不同就停。",
    "[底稿官方 2] `NUM_EPOCHS` 維持 prereg01 的 5（SMOKE 是 prereg01 冒煙的 1），用來比對 `state.pt`。新增 `EXTRA_EPOCHS`＝4（SMOKE 是 1）、`PEAK_LR`＝`LEARNING_RATE`＝1e-3、`FINAL_EPOCH`＝`NUM_EPOCHS`＋`EXTRA_EPOCHS`。`CKPT_DIR` 由設定格決定（`prereg02/rank/ckpt`），`makedirs` 前先過 `guard_write`。最後的 `print` 多印 GPU 型號和接著訓練的 epoch 數。正式執行沒有 GPU 就在這裡停下（收尾開關打開時例外，缺的逐盤輸出另由 dump 工具擋下）；有 GPU 但型號（環境紀錄格 nvidia-smi 查到的）不是 G4（RTX PRO 6000）或 A100 也停下（PREREG_02 第 7 節第 8 條）。",
    "[底稿分片工具] `restore_features(fp, src)` 改成可以指定來源，對來源只讀；還原到本機前 `guard_write(FEAT_DIR)`。`backup_features` 只寫到 `prereg02/features_rank`，`rmtree` 和每個複製目的地先過 `guard_write`。",
    "[底稿官方 9] 依序試 `prereg02/features_rank`、`prereg01/features_rank` 還原分片；都不能用才重抽（抽完備份到 prereg02）。`rmtree(FEAT_DIR)` 前先過 `guard_write`。`run_log.json` 另記分片來源。",
    "[新增 dump 工具] `d4(x, s)`（s＝0 原方向；s≥4 先在 dim 3 鏡射，再在 dims (2, 3) 旋轉 s%4 次，格內自我檢查 8 種兩兩不同且封閉）；`load_net()`（讀進記憶體、算 SHA-256、從同一份位元組載入）；inner 用分片的 `__key__`＝`等級_game_id` 對回 `df_val` 的列，assert 每盤剛好一次；本機驗證題照題目檔列順序和 slot 1..num_games 解析，失敗的保留位置（logits 填 NaN），每題一個 batch（和 prereg01 推論相同）；每個檔先寫 `.tmp` 再改名，附 `_DONE_<檔名>.json`（另記算它的那次執行的 GPU 型號、torch、cuDNN），重跑時對得上就跳過。正式執行時要算 dump 卻沒有 GPU 就停。另有 `record_p01_after()`（結尾指紋）、`pack_results()`（打包，跳過權重、暫存檔和 ≥ 100 MB 的檔）、`flush_drive()`（Colab 上 `drive.flush_and_unmount()`）。",
    "[新增 P1 輸出與重現核對] prereg01 epoch 1–4 原方向、epoch 5 八種對稱的逐盤 logits（inner、本機驗證題）和兩個 key 檔。重現核對三項：① epoch 5 在 inner 的逐盤 accuracy 和 `history.csv` 相差 ≤ 0.05 個百分點；② 每題 float32 平均機率和 prereg01 `rank_val_probs.csv` 最大差距 ≤ 1e-4（算 `val_p01e5_d4` 那次執行的 GPU 型號和 prereg01 不同時 1e-3；正式執行沒記到 GPU 就不通過）；③ 規則 B 不同的題 ≤ 4（SMOKE 依題數等比例縮小）。任一項不過就停，不訓練；停下前記結尾指紋、打包 `prereg02/rank_results_repro_failed.zip`、送出 Drive 的寫入。",
    "[新增接著訓練] 從 prereg01 `state.pt`（只讀、核對 SHA-256、設定、逐張量等於 `epoch_5.pth`）載入 model 和 AdamW 狀態，接著訓練 epoch 6–9：`LambdaLR` 每步 `max(0, 1 − t/T)`、T＝4×2,814，每個 epoch 開始 seed＝42＋epoch，其餘和 prereg01 相同。每個 epoch 存 `epoch_N.pth`、`state.pt`（含 scheduler、global_step、history、設定）、`history.csv`（inner 驗證只當監看）。有 prereg02 的 `state.pt` 就接續，設定不同就停。不產生 `best_model.pth`。訓練前用 `_count_w*.json` 和分片清單算出每個 epoch 的步數（webdataset 先依 worker 分分片），和 prereg01 的 2,814 步不同就停。收尾開關打開時不訓練，已完成的 epoch 從 prereg02 的 `state.pt` 讀回。",
    "[新增 P2 輸出] 用 `epoch_9.pth` 存 8 種對稱的逐盤 logits（inner、本機驗證題）；第 9 個 epoch 沒完成（包括收尾開關打開時）就跳過並記錄。",
    "[底稿結尾（改寫）] 改成核對所有 dump 的列數、形狀、NaN 位置和完成標記；記 `p01_readonly_after.json` 並和開頭比對（不同就在打包後停下）；補完 config／timings／run_log；打包 `prereg02/rank_results.zip`（跳過 `.pth`、`.pt`、`.tmp` 和 ≥ 100 MB 的檔；dump 一律 < 100 MB，否則在寫檔時就停）；最後 `drive.flush_and_unmount()`。",
    "[拿掉] prereg01 的設定紀錄與接續工具、官方 16（訓練迴圈）、18（訓練曲線）、20、21（推論）、所有官方 markdown 格。推論改由 dump 工具做，訓練改由接著訓練格做。",
    "[metadata] notebook metadata 的 Colab 設定拿掉預選的 A100 和「大量 RAM」（照第一格選 G4；拿不到 G4 時選 A100 標準版）。每格 cell id 是 `c10-rank02-NN`，cell metadata `c10` 記底稿的格號、底稿的 `c10` 和是否改動。"
]

# 和官方 notebook 的全部差異（prereg01 的差異表，沿用的格子照它）
DEVIATIONS = [
    "[①] 官方 cell 4：改用 `c10data.load_rows(..., part=\"fit\")` 讀訓練資料（5,767 人、720,307 盤），`part=\"inner\"` 讀內部驗證（640 人、73,556 盤），各自轉成 DataFrame。`load_rows` 會 assert 人數、盤數，並確認沒有任何一盤出現在本機驗證檔。結果存 `leak_check.json`（每次執行各記一筆）。SMOKE 時等 `load_rows` 全部通過，才用固定 seed 縮小資料。",
    "[②] 官方 cell 4：取代 `df.iloc[:VAL_SIZE]` 的切法：`df_train`、`df_val` 各自 `sample(frac=1.0, random_state=42)` 打散之後，才寫分片。",
    "[③] 官方 cell 2、8、9：`VAL_SIZE` 改成內部驗證實際寫入的盤數：每個抽特徵的 worker 寫完後，把 `rows`、`written`、`bad` 寫進分片目錄的 `_count_w<N>.json`；`extract_split` 加總、核對後回傳；cell 9 設定 `VAL_SIZE` 並印出。cell 2 先設成 `None`。",
    "[④] 官方 cell 20、21：`predict()` 改讀打包檔的 `local/rank_val_questions.csv`，拿掉 public／private 迴圈（內層程式因此少一層縮排，進度訊息不印 split 名稱）。每題另存 10 維平均機率到 `rank_val_probs.csv`（完整 float；沒有可解析棋譜的題 `n_parsed=0`、機率欄留空）。官方的 argmax 結果改存成 `pred_official_argmax.csv`（官方是 `./submission_rank.csv`）。`predict()` 因此多了題目檔、機率檔路徑和 `limit` 三個參數；SMOKE 時只推論前 12 題。",
    "[營運性] 新增：最前面兩格 markdown：這份 notebook 的用途與操作說明，以及這張差異表。",
    "[營運性] 新增：新增的程式格：設定（Colab 或本機、Drive 路徑、SMOKE 縮小設定、結果紀錄工具）、準備（解壓打包檔並逐檔核對 SHA-256、確認沒有答案檔、解開訓練資料、把打包檔的 `tools/`、`tutorial/` 加進 import 路徑）、環境紀錄（`env.json`）、分片備份工具、設定紀錄與接續工具、結尾核對，以及把結果目錄裡權重以外的檔案打成 `rank_results.zip`（放在結果目錄旁，方便下載）。",
    "[營運性] 新增（準備格）：缺 `sgfmill` 或 `webdataset` 時才用 pip 安裝，版本固定為 `sgfmill==1.1.1`、`webdataset==1.0.2`（本機冒煙測試用的版本）。官方 notebook 只列出需要的套件，不安裝也不固定版本。Colab 內建的 torch、pandas、numpy、matplotlib 不動，實際版本記在 `env.json`。",
    "[營運性] 新增（設定格）：在 Linux 上明確指定 multiprocessing 用 `fork`。官方的 worker 函式定義在 notebook 內，依賴 fork；Python 3.14 起 Linux 預設改成 forkserver。3.13 以前本來就是 fork，行為不變。實際的啟動方式記在 `config.json`。",
    "[營運性] metadata：notebook metadata 加上 Colab 設定（GPU、A100、高 RAM），其餘沿用官方。官方格沿用官方的 cell id，新增的格用 `c10-rank-NN`；每格的 cell metadata 加 `c10` 對照欄（對應的官方格、是否改動）。官方原本就沒有存輸出。",
    "[營運性] 官方 cell markdown 0、3、5、19、22：官方的說明原文不動，在後面附加一段「**本版**」說明：路徑由設定格決定（0）、① ② 的讀法（3）、分片會備份到 Drive（5）、④ 只預測本機驗證題（19）、分數不在這裡算（22）。",
    "[營運性] 全部：工作目錄切到 `WORK`（Colab 是 `/content/c10`），官方的相對路徑 `./training`、`./rank-features` 照用。webdataset 會把 Windows 路徑的 `C:` 當成網址格式，本機冒煙測試只能用相對路徑。",
    "[營運性] 全部：改過的格在開頭或行內加了說明註解（標 ①–④ 或「本版」）。",
    "[營運性] 全部：紀錄檔都是每次執行各記一筆、不覆蓋前幾次：`timings.json`（各階段秒數）、`run_log.json`（每次執行與接續）、`env.json`、`leak_check.json`、`config.json`（頂層是固定說明與跨執行的累計狀態，每次執行的實際設定在 `runs`）。JSON 和 CSV 的 float 都寫完整精度。原生 Windows 的正式執行會直接報錯，因為官方的多進程依賴 Linux 的 fork。",
    "[營運性] 官方 cell 2：`CKPT_DIR` 改到 Drive 的結果目錄 `ckpt/`，拿掉 `TEST_DIR`。`NPROC` 從寫死的 8 改成 `os.cpu_count()`（官方註解寫「設成 CPU 核數」）。加上 SMOKE 縮小設定（只在 SMOKE 生效）。最後的 `print` 多印 `NPROC`、batch、epoch 數。",
    "[營運性] 官方 cell 4：不印官方的「全量 N 局」和全量的等級分布；改成逐一印 `fit`、`inner` 的人數與盤數，等級分布只印訓練集（`df_train`）。顯示前 3 列時不印棋譜欄，不讓棋譜內容進 notebook 輸出。",
    "[營運性] 官方 cell 6：示範用的第一盤改取 `df_train`（本版沒有合併的 `df`）。",
    "[營運性] 官方 cell 9：分片特徵抽完後備份到 Drive 的 `features_rank/`，全部核對後才寫完成標記。重跑時有完整且設定相符的備份，就複製回來、核對 SHA-256，不重抽；備份缺檔、大小或 SHA-256 不符時改成重抽。重抽前先清掉不完整的舊分片。",
    "[營運性] 官方 cell 11、12：`pick_workers` 另外記下讀出的筆數（寫進 `config.json`）。本機冒煙測試的 worker 候選數是 0（Windows 的 spawn 無法 pickle notebook 內定義的函式）。多 worker 對不上時照官方退回 0，並用 0 個 worker 再數一次；還是對不上 `VAL_SIZE` 就停下來，因為這表示 ③ 沒有成立。",
    "[營運性] 官方 cell 16：每個 epoch 結束，把 model、optimizer、epoch、最佳分數、history 存成 `ckpt/state.pt`。重跑時從最後完成的 epoch 接續；`state.pt` 讀不進來、或設定和上次不同，就停下來並說明處理方式。另外記最佳 epoch 的編號，結束時一併印出。history 多記 loss、步數、樣本數、秒數和第幾次執行。官方的 `epoch_N.pth`、`best_model.pth`、`history.csv` 照樣存在 `ckpt/`。",
    "[營運性] 官方 cell 16：`sync_best()` 把最佳 epoch 的 `epoch_N.pth` 複製成 `best_model.pth`：接續時做一次（中斷時可能留下寫一半的 `best_model.pth`），正常訓練結束時也做一次。複製來源和官方存的 `best_model.pth` 是同一個 epoch 結束時的同一份權重。找不到那個 `epoch_N.pth` 就停下來並說明處理方式。",
    "[營運性] 官方 cell 16：每次執行記下有沒有讀 `state.pt`、是否接續訓練、實際訓練了哪些 epoch（`run_log.json`、`config.json` 的 `runs`），頂層的 `resumed` 表示是否曾經接續訓練。",
    "[營運性] 官方 cell 18：訓練曲線另存 `ckpt/history.png`。非 Colab 時用不開視窗的 Agg 後端，不 `show()`，畫完就 `close()`。",
    "[營運性] 官方 cell 20：`predict()` 讀題目檔時 `question_id` 以字串讀入（`dtype={'question_id': str}`），代號原樣照抄，不會被轉成數字。每題 assert 非空的 `sgf_` 欄數等於 `num_games`，題目檔格式不對就停。兩者都不改變預測。",
    "[營運性] 官方 cell 21：推論用的權重、題目檔、題數寫進 `config.json`，秒數寫進 `timings.json`；輸出路徑指向 Drive 的結果目錄。"
]


def write_json(path, obj):
    # 先寫暫存檔再改名，中斷時不會留下寫一半的 JSON；float 用 repr，不四捨五入
    guard_write(path)   # 本版：寫進 prereg01 目錄的一律擋下
    tmp = path + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=1, default=str)
    os.replace(tmp, path)


def read_json(path, default=None):
    if not os.path.exists(path):
        return default
    with open(path, encoding='utf-8') as f:
        return json.load(f)


def read_runs(path, **init):
    # 每次執行各記一筆的紀錄檔（{'runs': [...], ...}），不覆蓋前幾次；舊格式（整份只有一次執行）當成一筆保留
    obj = read_json(path)
    if obj is None:
        return {'runs': [], **init}
    if not isinstance(obj, dict) or not isinstance(obj.get('runs'), list):
        return {'runs': [{'legacy': obj}], **init}
    return obj


# 每次「全部執行」算一次 run；接續次數和時間記在 run_log.json
RUN_LOG_PATH = os.path.join(RESULT_DIR, 'run_log.json')
run_log = read_runs(RUN_LOG_PATH, resume_count=0)
RUN = {'run': len(run_log['runs']) + 1, 'start': now_iso(), 'end': None, 'smoke': SMOKE,
       'in_colab': IN_COLAB, 'features': None, 'start_epoch': None, 'loaded_state': False,
       'resumed_training': False, 'epochs_trained': []}
run_log['runs'].append(RUN)


def save_run_log():
    write_json(RUN_LOG_PATH, run_log)


TIMINGS_PATH = os.path.join(RESULT_DIR, 'timings.json')
timings = read_runs(TIMINGS_PATH)
TIMING = {'run': RUN['run'], 'start': RUN['start'], 'seconds': {}}
timings['runs'].append(TIMING)


def mark_time(stage, seconds):
    TIMING['seconds'][stage] = seconds
    write_json(TIMINGS_PATH, timings)


# config.json：頂層放固定的說明和跨執行的累計狀態；每次執行的實際設定各記一筆在 runs，不覆蓋前幾次
CONFIG_PATH = os.path.join(RESULT_DIR, 'config.json')
_old_config = read_runs(CONFIG_PATH)
CONFIG = {k: v for k, v in _old_config.items() if k != 'runs'}
CONFIG.update({
    'layout': '頂層：固定說明與跨執行的累計狀態；runs：每次執行各一筆的實際設定',
    'smoke': SMOKE, 'smoke_cfg': SMOKE_CFG if SMOKE else None,
    'prereg': 'docs/PREREG_02_OPENING_FUSION.md 第 1.1 節（競賽二）',
    'base_notebook': {'path': 'colab/prereg01_rank.ipynb', 'sha256': '5f97d51978674bcc789493a47f278f186fb839f3f2d396aa2d218a6820e9744b'},
    'changes_from_prereg01': P02_CHANGES,
    'smoke_cfg_p02': SMOKE_CFG_P02 if SMOKE else None,
    'p01_root': P01_ROOT,
    'official_notebook': 'third_party/aicup2026_tutorial/rank-prediction.ipynb',
    'deviations_from_official': DEVIATIONS,
    # resumed：是否曾有任何一次執行從 ckpt/state.pt 接續訓練。和 run_log.json 的 resume_count 同一個來源；
    # 已經訓練完、只重跑推論的那次不算
    'resumed': run_log['resume_count'] > 0, 'resume_count': run_log['resume_count'],
})
CONFIG['runs'] = _old_config['runs']
CONFIG_RUN = {'run': RUN['run'], 'start': RUN['start'], 'end': None, 'in_colab': IN_COLAB,
              'mp_start_method': MP_START_METHOD}
CONFIG['runs'].append(CONFIG_RUN)


def save_config(**kw):
    # 這次執行的實際設定：只更新 runs 裡這次的那一筆
    CONFIG_RUN.update(kw)
    write_json(CONFIG_PATH, CONFIG)


def save_config_summary(**kw):
    # 跨執行的累計狀態（是否曾接續、已完成幾個 epoch、最佳 epoch）：放頂層
    CONFIG.update(kw)
    write_json(CONFIG_PATH, CONFIG)


save_run_log()
save_config()
print('IN_COLAB', IN_COLAB, '| SMOKE', SMOKE, '| run', RUN['run'], '| multiprocessing', MP_START_METHOD)
print('WORK', WORK)
print('結果目錄', RESULT_DIR)
print('prereg01（只讀）', P01_ROOT)

In [ ]:
# 本版新增（preflight）：開跑前確認 prereg01 的權重、state.pt、機率檔都在，記下 SHA-256 和 prereg01 目錄的指紋。
# 任何一項不符就用中文說明處理方式後停下，還沒開始算（但執行階段開著就會扣運算單元）。
import hashlib

t0 = time.time()
P01_EPOCHS = SMOKE_CFG['num_epochs'] if SMOKE else 5
# 正式執行：這幾個檔必須和本機 commit 的 prereg01 結果（results/prereg01/rank/）逐位元組相同
P01_EXPECTED_SHA256 = {} if SMOKE else {
        "rank_val_probs.csv": "92cc750cad9eb114541047cf0c4863634ef4de0be1bac75f889edcee4823800c",
        "ckpt/history.csv": "d76272073a80efde206dc8cdac0d5f297aaeb23240f52ff05be35a673718cf35",
        "env.json": "2461a1daa8fef88a636716af8c39a82c48413e98c9eb2d0cccfe99ebe4d90b85"
    }
P01_INNER_ACC_REGISTERED = 25.1482   # PREREG_02 1.1 第 2 點；SMOKE 不用
P01_STEPS_REGISTERED = 2814          # prereg01 每個 epoch 的步數；SMOKE 不用
MIN_DRIVE_FREE_BYTES = 3 * 10 ** 9
P01_HELP = ('處理方式：確認這個 Colab 用的是跑 prereg01 的同一個 Google 帳號，而且 Drive 上有 '
            'MyDrive/C10/results/prereg01/rank/ckpt/（epoch_1..5.pth、best_model.pth、state.pt、history.csv）'
            '和 rank/rank_val_probs.csv、rank/env.json。不要搬動、改名或修改 prereg01 的任何檔案。'
            '還是不對就把這段訊息截圖給 Claude。')


def sha256_path(path):
    d = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            d.update(chunk)
    return d.hexdigest()


def gpu_name_of(env_run):
    # env.json 一筆執行紀錄裡的 GPU 型號（nvidia-smi 的第一欄）；沒有 GPU 時是 None
    smi = (env_run.get('gpu') or {}).get('nvidia_smi')
    return smi[0].split(',')[0].strip() if smi else None


P01_NEED = (['ckpt/epoch_%d.pth' % e for e in range(1, P01_EPOCHS + 1)]
            + ['ckpt/best_model.pth', 'ckpt/state.pt', 'ckpt/history.csv', 'rank_val_probs.csv', 'env.json'])
_missing = [n for n in P01_NEED if not os.path.isfile(os.path.join(P01_RANK, n))]
if _missing:
    raise SystemExit('找不到 prereg01 的檔案：%s（在 %s 底下）。%s' % (_missing, P01_RANK, P01_HELP))
P01_SHA = {n: sha256_path(os.path.join(P01_RANK, n)) for n in P01_NEED}
if P01_SHA['ckpt/best_model.pth'] != P01_SHA['ckpt/epoch_%d.pth' % P01_EPOCHS]:
    raise SystemExit('prereg01 的 best_model.pth 和 epoch_%d.pth 不是同一份權重（登記寫的是第 %d 個 epoch）。%s'
                     % (P01_EPOCHS, P01_EPOCHS, P01_HELP))
_bad = sorted(n for n, v in P01_EXPECTED_SHA256.items() if P01_SHA[n] != v)
if _bad:
    raise SystemExit('Drive 上 prereg01 的 %s 和本機 commit 的 prereg01 結果不同（SHA-256 不符）。%s' % (_bad, P01_HELP))

with open(os.path.join(P01_CKPT, 'history.csv'), newline='', encoding='utf-8') as f:
    P01_HISTORY = list(csv.DictReader(f))
if [int(h['epoch']) for h in P01_HISTORY] != list(range(1, P01_EPOCHS + 1)):
    raise SystemExit('prereg01 的 history.csv 不是 epoch 1..%d。%s' % (P01_EPOCHS, P01_HELP))
P01_INNER_ACC = float(P01_HISTORY[-1]['val_acc'])   # 重現核對 ① 的目標（最後一個 epoch 的 inner 逐盤 accuracy）
P01_STEPS = int(P01_HISTORY[-1]['steps'])           # 學習率排程的每個 epoch 步數
if not SMOKE and (abs(P01_INNER_ACC - P01_INNER_ACC_REGISTERED) > 1e-3 or P01_STEPS != P01_STEPS_REGISTERED):
    raise SystemExit('prereg01 的 history.csv 和登記的數字不同（inner accuracy %.4f%%、每個 epoch %d 步；登記是 %.4f%%、%d 步）。%s'
                     % (P01_INNER_ACC, P01_STEPS, P01_INNER_ACC_REGISTERED, P01_STEPS_REGISTERED, P01_HELP))
_p01_env_run = read_json(os.path.join(P01_RANK, 'env.json'))['runs'][-1]
P01_GPU_NAME = gpu_name_of(_p01_env_run)    # 重現核對 ② 的 GPU 規則用（和寫出逐盤輸出那次執行的 GPU 比）
P01_TORCH, P01_CUDNN = _p01_env_run.get('torch'), _p01_env_run.get('cudnn')

P01_FEAT_COMPLETE = os.path.exists(os.path.join(P01_FEAT, '_COMPLETE.json'))
if not P01_FEAT_COMPLETE:
    print('注意：prereg01 沒有完整的分片備份（%s）。如果 prereg02 也沒有，後面會重抽特徵，約多 6 分鐘。' % P01_FEAT)

try:
    DRIVE_FREE_BYTES = shutil.disk_usage(DRIVE_DIR).free
except OSError as e:
    DRIVE_FREE_BYTES = None
    print('注意：查不到 Drive 剩餘空間（%s: %s），請自己確認還有 3 GB 以上' % (type(e).__name__, e))
if DRIVE_FREE_BYTES is not None and DRIVE_FREE_BYTES < MIN_DRIVE_FREE_BYTES:
    raise SystemExit('Drive 只剩 %.2f GB，這份 notebook 要 3 GB 以上（權重和狀態約 0.75 GB，重抽特徵的話再加約 1.3 GB）。'
                     '處理方式：清出空間（Drive 的垃圾桶也要清空）後再按一次「全部執行」。' % (DRIVE_FREE_BYTES / 1e9))

# prereg01 目錄的指紋（開頭）。重跑時必須和第一次執行記下的相同：不同代表 prereg01 在兩次執行之間被改過
P01_BEFORE = p01_fingerprint()
P01_BEFORE_PATH = os.path.join(RESULT_DIR, 'p01_readonly_before.json')
_before_log = read_runs(P01_BEFORE_PATH)
if _before_log['runs'] and _before_log['runs'][0]['files'] != P01_BEFORE:
    _first = _before_log['runs'][0]['files']
    _changed = sorted(k for k in set(_first) | set(P01_BEFORE) if _first.get(k) != P01_BEFORE.get(k))
    raise SystemExit('prereg01 的目錄和第一次執行時不同（%d 個檔：%s）。prereg01 必須只讀，先查清楚是誰改的。'
                     '處理方式：不要再執行，把這段訊息截圖給 Claude。' % (len(_changed), _changed[:10]))
_before_log['runs'].append({'run': RUN['run'], 'time': now_iso(), 'root': P01_ROOT, 'n_files': len(P01_BEFORE),
                            'total_bytes': sum(v[0] for v in P01_BEFORE.values()), 'files': P01_BEFORE})
write_json(P01_BEFORE_PATH, _before_log)

save_config(preflight={'p01_root': P01_ROOT, 'p01_sha256': P01_SHA, 'p01_expected_sha256': P01_EXPECTED_SHA256,
                       'p01_inner_acc': P01_INNER_ACC, 'p01_steps_per_epoch': P01_STEPS, 'p01_gpu_name': P01_GPU_NAME,
                       'p01_torch': P01_TORCH, 'p01_cudnn': P01_CUDNN,
                       'p01_features_backup_complete': P01_FEAT_COMPLETE, 'drive_free_bytes': DRIVE_FREE_BYTES,
                       'p01_files_fingerprinted': len(P01_BEFORE)})
mark_time('preflight', time.time() - t0)
print('preflight 通過：prereg01 的 %d 個檔都在，best_model.pth＝epoch_%d.pth，inner accuracy 目標 %.4f%%，每個 epoch %d 步，'
      'prereg01 的 GPU：%s' % (len(P01_NEED), P01_EPOCHS, P01_INNER_ACC, P01_STEPS, P01_GPU_NAME))

In [ ]:
# 準備：缺套件才安裝、把打包檔複製到 WORK 解壓並逐檔核對 SHA-256、解開訓練資料
import hashlib
import importlib.util
import subprocess
import tarfile
import zipfile

t0 = time.time()
if IN_COLAB:
    # 版本固定成本機冒煙測試用的版本
    need = [pkg for mod, pkg in (('sgfmill', 'sgfmill==1.1.1'), ('webdataset', 'webdataset==1.0.2'))
            if importlib.util.find_spec(mod) is None]
    if need:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *need])

BUNDLE_NAME = 'C10_bundle_v1'
BUNDLE_ZIP_SRC = os.path.join(DRIVE_DIR, BUNDLE_NAME + '.zip')
BUNDLE_ZIP = os.path.join(WORK, BUNDLE_NAME + '.zip')
BUNDLE = os.path.join(WORK, BUNDLE_NAME)


def sha256_file(path):
    d = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            d.update(chunk)
    return d.hexdigest()


def bundle_mismatches(manifest):
    return [k for k, v in manifest['files'].items()
            if not os.path.exists(os.path.join(BUNDLE, k)) or sha256_file(os.path.join(BUNDLE, k)) != v]


with zipfile.ZipFile(BUNDLE_ZIP_SRC) as z:
    MANIFEST = json.loads(z.read(BUNDLE_NAME + '/bundle_manifest.json'))

if read_json(os.path.join(BUNDLE, 'bundle_manifest.json')) == MANIFEST and not bundle_mismatches(MANIFEST):
    print('打包檔已解壓，SHA-256 全部相符，跳過解壓')
else:
    shutil.rmtree(BUNDLE, ignore_errors=True)
    shutil.copyfile(BUNDLE_ZIP_SRC, BUNDLE_ZIP)
    with zipfile.ZipFile(BUNDLE_ZIP) as z:
        z.extractall(WORK)
    bad = bundle_mismatches(MANIFEST)
    if bad:
        raise SystemExit('打包檔 SHA-256 不符：%s' % bad)
    print('打包檔解壓完成，%d 個檔案的 SHA-256 全部相符' % len(MANIFEST['files']))

# 紀律 7：Colab 上不能有答案檔
_all_files = [os.path.join(r, n) for r, _d, ns in os.walk(BUNDLE) for n in ns]
assert not any('answers' in p for p in _all_files), '打包檔裡有答案檔，停止'

# 訓練資料：tar.xz 解開後是 ./training/train_<等級>.csv（另有 macOS 的 ._train_*.csv，load_rows 不會讀）
ARCHIVE = os.path.join(BUNDLE, 'training', 'aicup2026_go_training.tar.xz')
ARCHIVE_SHA = MANIFEST['files']['training/aicup2026_go_training.tar.xz']
TRAIN_FILES = ['train_%s.csv' % lv for lv in ('D', 'C', 'B', 'A', '1D', '2D', '3D', '4D', '5D', '6D')]
_marker = os.path.join('training', '.c10_extracted.json')
if (read_json(_marker, {}).get('archive_sha256') == ARCHIVE_SHA
        and all(os.path.exists(os.path.join('training', n)) for n in TRAIN_FILES)):
    print('訓練資料已解開，跳過')
else:
    shutil.rmtree('training', ignore_errors=True)
    with tarfile.open(ARCHIVE, 'r:xz') as tf:
        if hasattr(tarfile, 'data_filter'):
            tf.extractall(WORK, filter='data')
        else:
            tf.extractall(WORK)
    missing = [n for n in TRAIN_FILES if not os.path.exists(os.path.join('training', n))]
    if missing:
        raise SystemExit('解開後缺檔：%s' % missing)
    write_json(_marker, {'archive_sha256': ARCHIVE_SHA,
                         'bytes': {n: os.path.getsize(os.path.join('training', n)) for n in TRAIN_FILES}})
    print('訓練資料解開完成')

# 打包檔裡的 c10data.py、官方 utils.py／network.py 加進 import 路徑
for _sub in ('tools', 'tutorial'):
    _p = os.path.join(BUNDLE, _sub)
    if _p not in sys.path:
        sys.path.insert(0, _p)
LOCAL_DIR = os.path.join(BUNDLE, 'local')                      # 只有題目檔和候選庫
SPLIT_CSV = os.path.join(BUNDLE, 'split_v1', 'players_split.csv')
RANK_QUESTIONS = os.path.join(LOCAL_DIR, 'rank_val_questions.csv')
mark_time('prepare_bundle', time.time() - t0)
print('本機題目檔', sorted(os.listdir(LOCAL_DIR)))

In [ ]:
import glob
import os
import time
from multiprocessing import Pool, Process

import numpy as np
import pandas as pd
import torch
import torch.optim as optim
import webdataset as wds
from torch.utils.data import DataLoader

from utils import SGFParseRankPrediction
from network import GoRankResNet

print('torch', torch.__version__, '| cuda', torch.version.cuda,
      '| webdataset', wds.__version__)

In [ ]:
# 環境紀錄：每次執行各記一筆到結果目錄的 env.json（接續時可能換到不同的機器）
import platform
from importlib import metadata


def _pkg_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


def _ram_bytes():
    try:
        import psutil
        return psutil.virtual_memory().total
    except ImportError:
        pass
    if os.path.exists('/proc/meminfo'):
        with open('/proc/meminfo') as f:
            for line in f:
                if line.startswith('MemTotal:'):
                    return int(line.split()[1]) * 1024
    return None


_gpu = None
if torch.cuda.is_available():
    # 用 nvidia-smi 查，不在這裡初始化 CUDA（官方要到建模型時才初始化，之後才 fork 抽特徵的進程）
    try:
        _smi = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                               '--format=csv,noheader'], capture_output=True, text=True).stdout.strip().splitlines()
    except FileNotFoundError:
        _smi = None
    _gpu = {'nvidia_smi': _smi, 'device_count': torch.cuda.device_count()}
ENV_INFO = {
    'run': RUN['run'], 'start_time': RUN['start'], 'in_colab': IN_COLAB, 'smoke': SMOKE,
    'python': sys.version, 'platform': platform.platform(),
    'torch': torch.__version__, 'cuda': torch.version.cuda,
    'cudnn': torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else None,
    'gpu': _gpu, 'cpu_count': os.cpu_count(), 'ram_bytes': _ram_bytes(),
    'multiprocessing_start_method': MP_START_METHOD,
    'packages': {n: _pkg_version(n) for n in ('pandas', 'numpy', 'webdataset', 'sgfmill', 'matplotlib')},
    'bundle_zip_bytes': os.path.getsize(BUNDLE_ZIP_SRC), 'bundle_manifest': MANIFEST,
}
ENV_PATH = os.path.join(RESULT_DIR, 'env.json')
_env_log = read_runs(ENV_PATH)
_env_log['runs'].append(ENV_INFO)
write_json(ENV_PATH, _env_log)
print(json.dumps({k: ENV_INFO[k] for k in ('torch', 'cuda', 'gpu', 'cpu_count', 'ram_bytes', 'packages')},
                 ensure_ascii=False))

In [ ]:
# 路徑：官方的相對路徑照用（工作目錄已切到 WORK）；checkpoint 改存到 Drive 的結果目錄，中斷後才能接續
TRAIN_DIR = './training'
FEAT_DIR = './rank-features'
CKPT_DIR = RESULT_CKPT

# 等級由弱到強。順序很重要：評分時的「相鄰一級」就是這個順序上的相鄰。
LEVELS = ['D', 'C', 'B', 'A', '1D', '2D', '3D', '4D', '5D', '6D']
RANK_TO_ID = {lv: i for i, lv in enumerate(LEVELS)}
ID_TO_RANK = {i: lv for lv, i in RANK_TO_ID.items()}

NPROC = os.cpu_count() # 特徵抽取的進程數，設成 CPU 核數
PER_SHARD = 1000       # 每個 webdataset 分片的樣本數
SEED = 42
VAL_SIZE = None        # ③ 第 1 節寫完分片後，改成內部驗證實際寫入的盤數

BATCH_SIZE = 256
LEARNING_RATE = 1e-3
NUM_EPOCHS = 5
NUM_CLASSES = len(LEVELS)

if SMOKE:
    # 只在本機冒煙測試生效：縮小資料量與步數，邏輯不變（縮小方式見第一個設定格的 SMOKE_CFG）
    NPROC = SMOKE_CFG['nproc']
    PER_SHARD = SMOKE_CFG['per_shard']
    BATCH_SIZE = SMOKE_CFG['batch_size']
    NUM_EPOCHS = SMOKE_CFG['num_epochs']

# 本版（prereg02）：NUM_EPOCHS 維持 prereg01 的值，用來比對 prereg01 的 state.pt；接著訓練 EXTRA_EPOCHS 個 epoch
EXTRA_EPOCHS = SMOKE_CFG_P02['extra_epochs'] if SMOKE else 4
PEAK_LR = LEARNING_RATE            # 1e-3，每步線性降到 0
FINAL_EPOCH = NUM_EPOCHS + EXTRA_EPOCHS   # 正式是 9：登記指定的 checkpoint 是 epoch_9.pth
if not SMOKE:
    assert NUM_EPOCHS == 5 and EXTRA_EPOCHS == 4 and PEAK_LR == 1e-3 and BATCH_SIZE == 256, 'prereg02 的設定被改動'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# 本版：正式執行一定要有 GPU（逐盤輸出和接著訓練在 CPU 上要好幾天）。收尾開關打開時可以沒有 GPU，
# 但逐盤輸出必須都已存好（缺的話 dump 工具會停下）
if not SMOKE and DEVICE.type != 'cuda' and not FINISH_WITHOUT_TRAINING:
    raise SystemExit('沒有 GPU，停在這裡（還沒開始算）。處理方式：選單「執行階段 → 變更執行階段類型」選 G4 GPU'
                     '（灰色時選 A100 GPU，「大量 RAM」關掉）→ 儲存，再按一次「全部執行」。')
# 本版：GPU 型號只接受 G4（RTX PRO 6000）或 A100（PREREG_02 第 7 節第 8 條）。Colab 在選的 GPU 沒空時可能改派別的 GPU。
# 型號用環境紀錄格 nvidia-smi 查到的（不在這裡初始化 CUDA）
GPU_MODELS_OK = ('RTX PRO 6000', 'A100')
GPU_MODEL_NOW = gpu_name_of(ENV_INFO)
if not SMOKE and DEVICE.type == 'cuda' and not any(k in (GPU_MODEL_NOW or '') for k in GPU_MODELS_OK):
    raise SystemExit('這次拿到的 GPU 是 %s，不是 G4（RTX PRO 6000）或 A100，停在這裡（還沒開始算）。處理方式：'
                     '選單「執行階段 → 中斷連線並刪除執行階段」，再「執行階段 → 變更執行階段類型」選 G4 GPU'
                     '（灰色時選 A100 GPU，「大量 RAM」關掉）→ 儲存，再按一次「全部執行」。' % GPU_MODEL_NOW)

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

guard_write(CKPT_DIR)
os.makedirs(CKPT_DIR, exist_ok=True)
print('device', DEVICE, '| GPU', GPU_MODEL_NOW, '| NPROC', NPROC, '| batch', BATCH_SIZE,
      '| epochs', NUM_EPOCHS, '| extra epochs', EXTRA_EPOCHS, '| final epoch', FINAL_EPOCH)
RANK_TO_ID

In [ ]:
# ① 只讀訓練玩家：經 c10data.load_rows 讀。它會 assert 人數、盤數，並確認沒有任何一盤出現在本機驗證檔，
#    不符就直接 raise。訓練用 part="fit"，內部驗證用 part="inner"（另外 640 位玩家，不是從同一批棋切）。
from c10data import EXPECTED, load_rows

t0 = time.time()
leak = {'run': RUN['run'], 'time': now_iso(), 'c10data_sha256': MANIFEST['files']['tools/c10data.py'],
        'local_files_checked': sorted(os.listdir(LOCAL_DIR)), 'parts': {}}
frames = {}
for part in ('fit', 'inner'):
    t1 = time.time()
    rows = load_rows(TRAIN_DIR, SPLIT_CSV, part=part, local_dir=LOCAL_DIR)
    frames[part] = pd.DataFrame(rows, columns=['player_id', 'game_id', 'rank', 'color', 'sgf_content'])
    del rows
    got = {'players': int(frames[part]['player_id'].nunique()), 'games': int(len(frames[part]))}
    assert got == EXPECTED[part], (part, got, EXPECTED[part])
    leak['parts'][part] = {
        'players': got['players'], 'games': got['games'], 'expected': EXPECTED[part],
        'count_assert': '通過', 'leak_check': '外洩檢查通過（0 盤出現在本機驗證檔）',
        'games_by_rank': {lv: int((frames[part]['rank'] == lv).sum()) for lv in LEVELS},
        'seconds': time.time() - t1}
    print('%s：%s 人、%s 盤，人數盤數 assert 通過，外洩檢查通過'
          % (part, format(got['players'], ','), format(got['games'], ',')), flush=True)
assert not set(frames['fit']['player_id']) & set(frames['inner']['player_id'])
leak['fit_inner_player_overlap'] = 0
df_train, df_val = frames.pop('fit'), frames.pop('inner')

if SMOKE:
    # 只在本機冒煙測試：load_rows 全部通過之後，才用固定 seed 縮小
    df_train = df_train.sample(n=SMOKE_CFG['fit_games'], random_state=SMOKE_CFG['sample_seed'])
    df_val = df_val.sample(n=SMOKE_CFG['inner_games'], random_state=SMOKE_CFG['sample_seed'])
    leak['smoke_subsample'] = {'fit_games': len(df_train), 'inner_games': len(df_val),
                               'seed': SMOKE_CFG['sample_seed']}

# ② 讀進來時是照等級、再照黑白排好的，官方的分片又照列順序寫；各自打散後才寫分片，否則每個 batch 只有一個等級
df_train = df_train.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
df_val = df_val.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
# 每次執行各記一筆，不覆蓋前幾次
_leak_path = os.path.join(RESULT_DIR, 'leak_check.json')
_leak_log = read_runs(_leak_path)
_leak_log['runs'].append(leak)
write_json(_leak_path, _leak_log)
mark_time('load_rows', time.time() - t0)

print('等級分布（訓練）', df_train['rank'].value_counts().to_dict())
print('切分：訓練 %s / 驗證 %s' % (format(len(df_train), ','), format(len(df_val), ',')))
df_train[['player_id', 'game_id', 'rank', 'color']].head(3)

In [ ]:
_parser = SGFParseRankPrediction()
_demo = df_train.iloc[0]['sgf_content']
_feat, _err = _parser.features_from_sgf_content(_demo)
print('一局棋的特徵形狀:', _feat.shape, _feat.dtype)

In [ ]:
def extract_worker(df_part, pattern, wid):
    """抽一份切片的特徵並寫成分片。每個進程獨立持有 parser 與 ShardWriter。"""
    parser = SGFParseRankPrediction()
    written = bad = 0
    t0 = time.time()
    with wds.ShardWriter(pattern, maxcount=PER_SHARD, verbose=0) as sink:
        for _, row in df_part.iterrows():
            feat, err = parser.features_from_sgf_content(row['sgf_content'])
            if err is not None:
                bad += 1
                continue
            sink.write({
                '__key__': '%s_%s' % (row['rank'], row['game_id']),
                'features.npy': feat,
                'label.txt': str(row['rank']),
                'rank_idx.txt': str(RANK_TO_ID[row['rank']]),
                'game_name.txt': str(row['game_id']),
            })
            written += 1
            if written % 20000 == 0:
                print('  [w%d] %d/%d  %.0f 分'
                      % (wid, written, len(df_part), (time.time() - t0) / 60),
                      flush=True)
    print('  [w%d] 寫入 %d 筆（解析失敗 %d）%.1f 分'
          % (wid, written, bad, (time.time() - t0) / 60), flush=True)
    # ③ 把實際寫入的筆數交回主程式（worker 是另一個進程，用小檔案回報）
    with open('%s/_count_w%d.json' % (os.path.dirname(pattern), wid), 'w') as f:
        json.dump({'rows': len(df_part), 'written': written, 'bad': bad}, f)


def extract_split(df_part, tag):
    """跑完一個 split，並確認產出真的有內容。"""
    outdir = f'{FEAT_DIR}/{tag}'
    os.makedirs(outdir, exist_ok=True)

    procs = []
    for i in range(NPROC):
        pat = '%s/%s-w%d-%%06d.tar.gz' % (outdir, tag, i)
        p = Process(target=extract_worker, args=(df_part.iloc[i::NPROC], pat, i))
        p.start()
        procs.append(p)
    for p in procs:
        p.join()
        if p.exitcode != 0:
            raise SystemExit('%s: worker 非正常結束（exitcode %s）' % (tag, p.exitcode))

    shards = glob.glob(f'{outdir}/*.tar.gz')
    total = sum(os.path.getsize(s) for s in shards)
    avg = total / len(shards) if shards else 0
    print('%s: %d 個分片, %.2f GB, 平均 %.1f KB/分片'
          % (tag, len(shards), total / 1e9, avg / 1024), flush=True)
    # 平均分片大小是最簡單的健全性檢查：寫入迴圈若出錯，計數器仍會累加、log 看起來
    # 正常，但 tar 是空的（gzip 後僅 46 bytes）。只數分片數量抓不到這種情況。
    if avg < 10 * 1024:
        raise SystemExit('%s: 分片平均僅 %.0f bytes，判定為空分片' % (tag, avg))

    # ③ 加總各 worker 回報的實際寫入筆數
    counts = [read_json('%s/_count_w%d.json' % (outdir, i)) for i in range(NPROC)]
    got = {k: sum(c[k] for c in counts) for k in ('rows', 'written', 'bad')}
    assert got['rows'] == len(df_part) and got['written'] + got['bad'] == got['rows'], got
    print('%s: 實際寫入 %d 筆（解析失敗 %d）' % (tag, got['written'], got['bad']), flush=True)
    return got

In [ ]:
# 分片備份工具：抽好的分片備份到 Drive；重跑時有完整且設定相符的備份就複製回來，不重抽
FEAT_MARKER = os.path.join(FEAT_BACKUP, '_COMPLETE.json')


def _feature_files(root):
    # {'train/xxx.tar.gz': bytes, ...}，包含各 worker 回報筆數的 _count_w*.json
    out = {}
    for tag in ('train', 'val'):
        d = os.path.join(root, tag)
        if os.path.isdir(d):
            for n in sorted(os.listdir(d)):
                out['%s/%s' % (tag, n)] = os.path.getsize(os.path.join(d, n))
    return out


def backup_features(fingerprint, counts):
    # 全部複製並核對大小之後，才寫完成標記。本版：只寫到 prereg02 的 FEAT_BACKUP
    guard_write(FEAT_BACKUP)
    shutil.rmtree(FEAT_BACKUP, ignore_errors=True)
    files = _feature_files(FEAT_DIR)
    for rel in files:
        dst = guard_write(os.path.join(FEAT_BACKUP, rel))
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copyfile(os.path.join(FEAT_DIR, rel), dst)
    if _feature_files(FEAT_BACKUP) != files:
        raise SystemExit('分片備份的檔案大小和本機不符')
    write_json(FEAT_MARKER, {
        'fingerprint': fingerprint, 'counts': counts, 'nproc_used': NPROC, 'time': now_iso(),
        'files': {rel: {'bytes': b, 'sha256': sha256_file(os.path.join(FEAT_DIR, rel))}
                  for rel, b in files.items()}})
    print('分片已備份到 %s（%d 個檔，%.2f GB）' % (FEAT_BACKUP, len(files), sum(files.values()) / 1e9))


def restore_features(fingerprint, src):
    # 回傳 {'counts':…, 'mode':…, 'source':…}；沒有可用的備份（沒有、設定不同、缺檔、壞檔）時回傳 None，呼叫端會試下一個來源或重抽
    # 本版：備份來源 src 可以指定（prereg02 或 prereg01 的 features_rank），對來源只讀；只寫本機的 FEAT_DIR
    guard_write(FEAT_DIR)
    try:
        m = read_json(os.path.join(src, '_COMPLETE.json'))
        if m is not None:
            want = {rel: v['bytes'] for rel, v in m['files'].items()}
            counts, same = m['counts'], m['fingerprint'] == fingerprint
    except (OSError, ValueError, KeyError, TypeError, AttributeError) as e:
        print('Drive 上的分片完成標記讀不進來（%s: %s），改成重抽' % (type(e).__name__, e))
        return None
    if m is None:
        print('%s：沒有完整的分片備份' % src)
        return None
    if not same:
        print('Drive 上的分片備份和這次的設定不同，不用')
        return None
    mode = 'local_reused'
    if _feature_files(FEAT_DIR) != want:
        if _feature_files(src) != want:
            print('Drive 上的分片備份缺檔或大小不符，改成重抽')
            return None
        shutil.rmtree(FEAT_DIR, ignore_errors=True)
        try:
            for rel in want:
                dst = os.path.join(FEAT_DIR, rel)
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                shutil.copyfile(os.path.join(src, rel), dst)
        except OSError as e:
            print('從 Drive 複製分片失敗（%s: %s），改成重抽' % (type(e).__name__, e))
            shutil.rmtree(FEAT_DIR, ignore_errors=True)
            return None
        mode = 'restored_from_drive'
    try:
        bad = [rel for rel, v in m['files'].items() if sha256_file(os.path.join(FEAT_DIR, rel)) != v['sha256']]
    except (OSError, KeyError, TypeError) as e:
        bad = ['（核對失敗：%s: %s）' % (type(e).__name__, e)]
    if bad:
        print('分片的 SHA-256 和備份紀錄不符（%d 個檔），改成重抽' % len(bad))
        shutil.rmtree(FEAT_DIR, ignore_errors=True)
        return None
    print('分片：%s（%d 個檔，來源 %s）' % (mode, len(want), src))
    return {'counts': counts, 'mode': mode, 'source': src}

In [ ]:
# 一次性作業。分片產出後重跑訓練可以直接從下一節開始。
# 本版：Drive 上有完整的分片備份就直接複製回來，不重抽；沒有（或缺檔、壞檔）的話先清掉不完整的舊分片再抽，抽完備份到 Drive
t0 = time.time()
FEAT_FINGERPRINT = {'bundle_files': MANIFEST['files'], 'smoke': SMOKE,
                    'smoke_cfg': SMOKE_CFG if SMOKE else None, 'seed': SEED, 'per_shard': PER_SHARD,
                    'fit_games': len(df_train), 'inner_games': len(df_val)}
# 本版：先試 prereg02 自己的備份，再試 prereg01 的備份（只讀）；兩個都不能用才重抽
restored = None
for _src in (FEAT_BACKUP, P01_FEAT):
    restored = restore_features(FEAT_FINGERPRINT, _src)
    if restored is not None:
        break
if restored is None:
    print('prereg02 和 prereg01 都沒有可用的分片備份：重抽（約多 6 分鐘），抽完備份到 %s' % FEAT_BACKUP)
    guard_write(FEAT_DIR)
    shutil.rmtree(FEAT_DIR, ignore_errors=True)
    val_counts = extract_split(df_val, 'val')
    train_counts = extract_split(df_train, 'train')
    mark_time('features_extract', time.time() - t0)
    t1 = time.time()
    backup_features(FEAT_FINGERPRINT, {'val': val_counts, 'train': train_counts})
    mark_time('features_backup', time.time() - t1)
    RUN['features'] = 'extracted'
else:
    val_counts, train_counts = restored['counts']['val'], restored['counts']['train']
    mark_time('features_restore', time.time() - t0)
    RUN['features'] = restored['mode']
    RUN['features_source'] = restored['source']
save_run_log()

VAL_SIZE = val_counts['written']   # ③ 內部驗證實際寫入的盤數
print('VAL_SIZE=%d（內部驗證實際寫入）；訓練實際寫入 %d 筆' % (VAL_SIZE, train_counts['written']))
print('完成，總耗時 %.1f 分' % ((time.time() - t0) / 60))

In [ ]:
def decode_sample(sample):
    return (torch.from_numpy(sample['features.npy']).float(),
            torch.tensor(int(sample['rank_idx.txt']), dtype=torch.long),
            sample['game_name.txt'])


def make_loader(shards, workers, shuffle):
    ds = wds.WebDataset(shards, shardshuffle=1 if shuffle else 0)
    if shuffle:
        ds = ds.shuffle(1000)
    ds = ds.decode().map(decode_sample).batched(BATCH_SIZE, partial=True)
    return DataLoader(ds, batch_size=None, num_workers=workers,
                      pin_memory=True, persistent_workers=workers > 0)


WORKER_CHECKS = []   # 每次數驗證集的結果，寫進 config.json


def pick_workers(val_shards, candidate=4):
    """數一遍驗證集：筆數對得上才用多 worker，否則退回單執行緒。"""
    total = sum(len(y) for _x, y, _n in make_loader(val_shards, candidate, False))
    WORKER_CHECKS.append({'candidate': candidate, 'counted': total, 'expected': VAL_SIZE,
                          'matched': total == VAL_SIZE})
    if total == VAL_SIZE:
        print('num_workers=%d 驗證通過（%d 筆）' % (candidate, total))
        return candidate
    print('num_workers=%d 讀出 %d 筆（應為 %d），退回 0' % (candidate, total, VAL_SIZE))
    return 0

In [ ]:
train_shards = sorted(glob.glob(f'{FEAT_DIR}/train/*.tar.gz'))
val_shards = sorted(glob.glob(f'{FEAT_DIR}/val/*.tar.gz'))
print('分片：train %d / val %d' % (len(train_shards), len(val_shards)))

# 本機冒煙測試（Windows 的 spawn）一律 0 個 worker；Colab 照官方先試 4 個
WORKER_CANDIDATE = SMOKE_CFG['loader_workers'] if (SMOKE and not IN_COLAB) else 4
t0 = time.time()
workers = pick_workers(val_shards, WORKER_CANDIDATE)
if workers == 0 and WORKER_CANDIDATE > 0:
    pick_workers(val_shards, 0)   # 照官方退回 0 之後，用 0 個 worker 再數一次
if not WORKER_CHECKS[-1]['matched']:
    raise SystemExit('驗證分片讀出 %d 筆，VAL_SIZE 是 %d：③ 沒有成立'
                     % (WORKER_CHECKS[-1]['counted'], VAL_SIZE))
mark_time('pick_workers', time.time() - t0)
train_loader = make_loader(train_shards, workers, True)
val_loader = make_loader(val_shards, workers, False)

_x, _y, _n = next(iter(val_loader))
print('一個 batch:', _x.shape, _y[:8].tolist())

In [ ]:
model = GoRankResNet(num_classes=NUM_CLASSES).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
criterion = torch.nn.CrossEntropyLoss()

print('參數量 {:,}'.format(sum(p.numel() for p in model.parameters())))
print('batch %d | lr %s | epochs %d' % (BATCH_SIZE, LEARNING_RATE, NUM_EPOCHS))

In [ ]:
def validate(model, loader, criterion):
    model.eval()
    loss_sum, correct, total, batches = 0.0, 0, 0, 0
    with torch.no_grad():
        for inputs, labels, _ in loader:
            inputs = inputs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            outputs = model(inputs)
            loss_sum += criterion(outputs, labels).item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            batches += 1
    model.train()
    return (loss_sum / batches if batches else 0.0,
            100 * correct / total if total else 0.0)

In [ ]:
# 本版新增：逐盤輸出（dump）工具
# - inner：對內部驗證的分片跑一遍（和 validate() 一樣 batch 256、不打散），用分片的 __key__＝「等級_game_id」對回 df_val 的列
# - 本機驗證題：照題目檔的列順序和 slot 1..num_games 解析，失敗的保留位置（logits 填 NaN）；
#   每題一個 batch，和 prereg01 的推論相同，重現核對才對得上
# - 每個檔先寫 .tmp 再改名，附 _DONE_<檔名>.json；重跑時完成標記、SHA-256、權重 SHA-256 都對得上就跳過
import io
import math
import random
import zipfile

DUMP_DIR = os.path.join(RESULT_DIR, 'dumps')
guard_write(DUMP_DIR)
os.makedirs(DUMP_DIR, exist_ok=True)
DUMP_MAX_BYTES = 100 * 1024 * 1024   # 結果 zip 會跳過 ≥ 100 MB 的檔，所以每個 dump 都必須更小
GPU_BATCH = 64                         # 和 prereg01 推論相同
NUM_SYMS = 8
EXPECTED_INNER_GAMES = SMOKE_CFG['inner_games'] if SMOKE else 73556
if not VAL_SIZE == len(df_val) == EXPECTED_INNER_GAMES:
    raise SystemExit('內部驗證的盤數不對：VAL_SIZE %s、df_val %d、應該是 %d。請截圖給 Claude。'
                     % (VAL_SIZE, len(df_val), EXPECTED_INNER_GAMES))


def d4(x, s):
    # 8 種對稱：s=0 原方向；s>=4 先在 dim 3 鏡射，再在 dims (2, 3) 旋轉 s%4 次（x 是 [batch, 500, 19, 19]）
    if s == 0:
        return x
    y = torch.flip(x, dims=(3,)) if s >= 4 else x
    return torch.rot90(y, k=s % 4, dims=(2, 3)).contiguous()


# 自我檢查：8 種兩兩不同，而且任兩種接連做仍在這 8 種之內（封閉，是一個群）
_t = torch.arange(2 * 19 * 19, dtype=torch.float32).reshape(1, 2, 19, 19)
_imgs = [d4(_t, s) for s in range(NUM_SYMS)]
assert all(not torch.equal(_imgs[a], _imgs[b]) for a in range(NUM_SYMS) for b in range(a))
assert all(any(torch.equal(d4(_imgs[a], b), c) for c in _imgs) for a in range(NUM_SYMS) for b in range(NUM_SYMS))


def load_net(path, expect_sha=None):
    # 整個檔讀進記憶體、算 SHA-256，再從同一份位元組載入：記下的 SHA 就是實際用的權重
    with open(path, 'rb') as f:
        raw = f.read()
    got = hashlib.sha256(raw).hexdigest()
    if expect_sha is not None and got != expect_sha:
        raise SystemExit('%s 的 SHA-256 和開頭記下的不同（執行中被改動？）。請把這段訊息截圖給 Claude。' % path)
    net = GoRankResNet(num_classes=NUM_CLASSES)
    net.load_state_dict(torch.load(io.BytesIO(raw), map_location='cpu'))
    net.eval().to(DEVICE)
    return net, got


def _dump_path(stem):
    return os.path.join(DUMP_DIR, stem + '.npz')


def _marker_path(stem):
    return os.path.join(DUMP_DIR, '_DONE_%s.json' % stem)


def dump_is_done(stem, rows, syms, weights_sha):
    m = read_json(_marker_path(stem))
    if not m or not os.path.exists(_dump_path(stem)):
        return False
    if (m.get('rows'), m.get('syms'), m.get('weights_sha256')) != (rows, syms, weights_sha):
        return False
    return sha256_path(_dump_path(stem)) == m.get('sha256')


def write_dump(stem, rows, syms, weights, weights_sha, seconds, arrays):
    # arrays：要存進 npz 的 {名稱: 陣列}；rows、syms、weights、weights_sha 記進完成標記
    # 舊的完成標記先拿掉 → 寫 .tmp → 檢查大小 → 改名 → 最後才寫完成標記
    path, marker = _dump_path(stem), _marker_path(stem)
    tmp = path + '.tmp'
    for p in (path, tmp, marker):
        guard_write(p)
    if os.path.exists(marker):
        os.remove(marker)
    with open(tmp, 'wb') as f:
        np.savez(f, **arrays)
    size = os.path.getsize(tmp)
    if size >= DUMP_MAX_BYTES:
        raise SystemExit('%s 有 %.1f MB，超過 100 MB（結果 zip 會漏掉它）。請截圖給 Claude。' % (stem, size / 1e6))
    os.replace(tmp, path)
    # gpu_name／torch／cudnn：算出這個 dump 的那次執行（重跑時可能換了 GPU；重現核對 ② 的 GPU 規則看這裡）
    write_json(marker, {'file': 'dumps/%s.npz' % stem, 'rows': rows, 'syms': syms, 'bytes': size,
                        'sha256': sha256_path(path), 'weights': weights, 'weights_sha256': weights_sha,
                        'seconds': seconds, 'time': now_iso(), 'run': RUN['run'],
                        'gpu_name': gpu_name_of(ENV_INFO), 'torch': torch.__version__, 'cudnn': ENV_INFO.get('cudnn'),
                        'device': str(DEVICE)})


def require_gpu_for(stem):
    # 正式執行不在 CPU 上算逐盤輸出（要好幾天）；收尾開關打開又沒有 GPU 時，缺的 dump 在這裡停下
    if not SMOKE and DEVICE.type != 'cuda':
        raise SystemExit('%s 還沒算好，但這個執行階段沒有 GPU，停在這裡。處理方式：選單「執行階段 → 變更執行階段類型」'
                         '選 G4 GPU（灰色時選 A100 GPU，「大量 RAM」關掉）→ 儲存，再按一次「全部執行」。' % stem)


def record_p01_after():
    # prereg01 目錄的指紋（結尾，或重現核對沒過時）：和這次開頭記的比，寫進 p01_readonly_after.json；回傳 (指紋, 不同的檔)
    after = p01_fingerprint()
    path = os.path.join(RESULT_DIR, 'p01_readonly_after.json')
    log = read_runs(path)
    changed = sorted(k for k in set(P01_BEFORE) | set(after) if P01_BEFORE.get(k) != after.get(k))
    log['runs'].append({'run': RUN['run'], 'time': now_iso(), 'root': P01_ROOT, 'n_files': len(after),
                        'total_bytes': sum(v[0] for v in after.values()), 'files': after,
                        'same_as_before': not changed, 'changed_files': changed})
    write_json(path, log)
    return after, changed


def pack_results(zip_name):
    # 結果目錄裡除了權重和暫存檔以外的檔案，打成一個 zip 放在結果目錄旁邊（prereg02/<zip_name>）；回傳 (路徑, 跳過的大檔)
    path = guard_write(os.path.join(os.path.dirname(RESULT_DIR), zip_name))
    tmp = guard_write(path + '.tmp')
    skipped = []
    with zipfile.ZipFile(tmp, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(RESULT_DIR):
            dirs.sort()
            for f in sorted(files):
                p = os.path.join(root, f)
                if f.endswith(('.pth', '.pt', '.tmp')):
                    continue   # 權重與暫存檔不帶
                if os.path.getsize(p) > 100 * 1024 * 1024:
                    skipped.append(os.path.relpath(p, RESULT_DIR))
                    continue
                zf.write(p, os.path.relpath(p, os.path.dirname(RESULT_DIR)))
    os.replace(tmp, path)
    if skipped:
        print('注意：這些檔超過 100 MB，沒有打包：%s' % skipped)
    return path, skipped


def flush_drive():
    # Colab：下載 zip、刪除執行階段之前，先把 Drive 的寫入全部送出再卸載（權重、state.pt、zip 才確定都在 Drive 上）
    if IN_COLAB:
        drive.flush_and_unmount()
        print('Drive 已寫入完成並卸載；要再跑請重新「全部執行」')


def write_keys(stem, arrays):
    # key 檔：已有完整的就讀回來比對，必須完全相同（不同代表列順序變了，已存的 dump 會對錯盤）
    rows = len(next(iter(arrays.values())))
    if dump_is_done(stem, rows, None, None):
        with np.load(_dump_path(stem), allow_pickle=False) as z:
            same = sorted(z.files) == sorted(arrays) and all(
                z[k].dtype == v.dtype and np.array_equal(z[k], v) for k, v in arrays.items())
        if not same:
            raise SystemExit('%s 和這次算出的 key 不同，已存的逐盤輸出可能對錯盤。處理方式：不要刪檔，把這段訊息截圖給 Claude。'
                             % stem)
        print('  %s：已存在且相同，跳過' % stem)
        return 'skipped'
    t0 = time.time()
    write_dump(stem, rows, None, None, None, time.time() - t0, arrays)
    print('  %s：寫出 %d 列' % (stem, rows))
    return 'written'


# inner 的 key：df_val 的列順序（load_rows 的順序再用 SEED 打散，每次執行都相同）
INNER_KEYS = {
    'rank': np.array(df_val['rank'].tolist(), dtype=str),
    'game_id': np.array(df_val['game_id'].tolist(), dtype=str),
    'player_id': np.array(df_val['player_id'].tolist(), dtype=str),
    'color': np.array(df_val['color'].tolist(), dtype=str),
    'label': np.array([RANK_TO_ID[r] for r in df_val['rank']], dtype=np.int8),
}
INNER_INDEX = {(r, g): i for i, (r, g) in enumerate(zip(df_val['rank'], df_val['game_id']))}
assert len(INNER_INDEX) == len(df_val), '(等級, game_id) 在內部驗證裡不唯一'
assert set(INNER_KEYS['color'].tolist()) <= {'B', 'W'}


def decode_with_key(sample):
    return (torch.from_numpy(sample['features.npy']).float(),
            torch.tensor(int(sample['rank_idx.txt']), dtype=torch.long),
            sample['game_name.txt'], sample['__key__'])


# inner 專用的 loader：和 val_loader 相同的分片、worker 數、batch，多回傳分片的 __key__
KEY_LOADER = DataLoader(
    wds.WebDataset(val_shards, shardshuffle=0).decode().map(decode_with_key).batched(BATCH_SIZE, partial=True),
    batch_size=None, num_workers=workers, pin_memory=True, persistent_workers=workers > 0)


def dump_inner(net, name, syms, weights, weights_sha):
    stem, n = 'inner_' + name, len(df_val)
    if dump_is_done(stem, n, syms, weights_sha):
        print('  %s：已完成，跳過' % stem)
        return 'skipped'
    require_gpu_for(stem)
    t0 = time.time()
    logits = np.full((n, syms, NUM_CLASSES), np.nan, dtype=np.float32)
    seen = np.zeros(n, dtype=bool)
    count = 0
    with torch.no_grad():
        for x, y, names, keys in KEY_LOADER:
            idx = []
            for lab, g, k in zip(y.tolist(), names, keys):
                if k != '%s_%s' % (LEVELS[lab], g):
                    raise SystemExit('分片的 __key__ %r 和（等級, game_id）＝（%s, %s）對不上。請截圖給 Claude。'
                                     % (k, LEVELS[lab], g))
                i = INNER_INDEX.get((LEVELS[lab], g))
                if i is None:
                    raise SystemExit('分片裡的 %s 不在這次的內部驗證（df_val）裡。請截圖給 Claude。' % k)
                idx.append(i)
            idx = np.asarray(idx, dtype=np.int64)
            if len(set(idx.tolist())) != len(idx) or seen[idx].any():
                raise SystemExit('內部驗證有盤出現兩次（%s）。請截圖給 Claude。' % stem)
            seen[idx] = True
            count += len(idx)
            xb = x.to(DEVICE, non_blocking=True)
            for s in range(syms):
                logits[idx, s] = net(d4(xb, s)).float().cpu().numpy()
    if count != n or not seen.all():
        raise SystemExit('%s：讀到 %d 盤，內部驗證是 %d 盤，有盤沒讀到。請截圖給 Claude。' % (stem, count, n))
    write_dump(stem, n, syms, weights, weights_sha, time.time() - t0,
               {'logits': logits, 'weights_sha256': np.array(weights_sha), 'syms': np.array(syms)})
    print('  %s：%d 盤 × %d 種方向，%.0f 秒' % (stem, n, syms, time.time() - t0), flush=True)
    return 'written'


_infer_parser = SGFParseRankPrediction()


def sgf_to_features(sgf_content):
    # 給 multiprocessing.Pool 用的頂層函式（和 prereg01 推論相同）
    feat, err = _infer_parser.features_from_sgf_content(sgf_content)
    return None if err is not None else feat


VAL = None   # parse_val() 的結果，整份 notebook 共用（只解析一次）


def parse_val():
    # 本機驗證題：題目檔的列順序、slot 1..num_games；SMOKE 時和 prereg01 冒煙一樣只取前幾題
    global VAL
    if VAL is not None:
        return VAL
    t0 = time.time()
    test = pd.read_csv(RANK_QUESTIONS, dtype={'question_id': str})
    if SMOKE:
        test = test.head(SMOKE_CFG['questions'])
    sgf_cols = [c for c in test.columns if c.startswith('sgf_')]
    qids, slots, colors, sgfs, spans = [], [], [], [], []
    for _, r in test.iterrows():
        # 和 prereg01 一樣取非空的 sgf_ 欄；本版另外確認它們剛好是 sgf_1..sgf_num_games，slot 才對得回執色
        n = int(r['num_games'])
        cols = [c for c in sgf_cols if pd.notna(r[c]) and str(r[c]).strip()]
        assert cols == ['sgf_%d' % j for j in range(1, n + 1)], (r['question_id'], cols, n)
        spans.append((r['question_id'], len(sgfs), n))
        for j in range(1, n + 1):
            c = r['color_%d' % j]
            assert c in ('B', 'W'), (r['question_id'], j, c)
            qids.append(r['question_id'])
            slots.append(j)
            colors.append(c)
            sgfs.append(r['sgf_%d' % j])
    with Pool(NPROC) as pool:
        feats = pool.map(sgf_to_features, sgfs, chunksize=8)
    parsed = np.array([f is not None for f in feats], dtype=bool)
    VAL = {'spans': spans, 'feats': feats,
           'keys': {'question_id': np.array(qids, dtype=str), 'slot': np.array(slots, dtype=np.int32),
                    'color': np.array(colors, dtype=str), 'parsed': parsed}}
    mark_time('parse_val', time.time() - t0)
    print('本機驗證題：%d 題、%d 盤，解析失敗 %d 盤，%.0f 秒'
          % (len(spans), len(feats), int((~parsed).sum()), time.time() - t0), flush=True)
    return VAL


def dump_val(net, name, syms, weights, weights_sha):
    V = parse_val()
    stem, n = 'val_' + name, len(V['feats'])
    if dump_is_done(stem, n, syms, weights_sha):
        print('  %s：已完成，跳過' % stem)
        return 'skipped'
    require_gpu_for(stem)
    t0 = time.time()
    logits = np.full((n, syms, NUM_CLASSES), np.nan, dtype=np.float32)
    with torch.no_grad():
        for _qid, start, k in V['spans']:
            ok = [start + j for j in range(k) if V['feats'][start + j] is not None]
            if not ok:
                continue
            # 和 prereg01 相同：一題的棋疊成一個 batch（每批最多 GPU_BATCH 盤）
            x = torch.from_numpy(np.stack([V['feats'][g] for g in ok])).float()
            for i in range(0, len(x), GPU_BATCH):
                xb = x[i:i + GPU_BATCH].to(DEVICE)
                rows = ok[i:i + GPU_BATCH]
                for s in range(syms):
                    logits[rows, s] = net(d4(xb, s)).float().cpu().numpy()
    write_dump(stem, n, syms, weights, weights_sha, time.time() - t0,
               {'logits': logits, 'weights_sha256': np.array(weights_sha), 'syms': np.array(syms)})
    print('  %s：%d 盤 × %d 種方向，%.0f 秒' % (stem, n, syms, time.time() - t0), flush=True)
    return 'written'


_E1 = math.exp(-1)


def _argmax_first(values):
    # 最大值的位置；同分取前面的（和 numpy.argmax、tools/decide.py 相同）
    best = 0
    for i in range(1, len(values)):
        if values[i] > values[best]:
            best = i
    return best


def rule_b(p):
    # 規則 B（和 tools/decide.py 的 rule_b 相同）：使 p_k + e^-1·(p_{k-1} + p_{k+1}) 最大的等級，兩端外側算 0
    n = len(p)
    return LEVELS[_argmax_first([p[k] + _E1 * ((p[k - 1] if k > 0 else 0.0) + (p[k + 1] if k < n - 1 else 0.0))
                                 for k in range(n)])]


print('dump 工具就緒：inner %d 盤（worker %d）' % (len(df_val), workers))

In [ ]:
# 本版新增：P1 逐盤輸出（prereg01 epoch 1–5）與重現核對（PREREG_02 1.1 第 2、3 點）
# 三項核對任一項不過就停在這裡：不接著訓練、不產生新的 checkpoint
t0 = time.time()
write_keys('inner_keys', INNER_KEYS)
write_keys('val_keys', parse_val()['keys'])

P1_DUMPS = [('p01e1', 1, 1), ('p01e2', 2, 1), ('p01e3', 3, 1), ('p01e4', 4, 1), ('p01e5_d4', 5, NUM_SYMS)]
DUMP_LOG = {}
for _name, _ep, _syms in P1_DUMPS:
    # SMOKE：prereg01 冒煙只訓練 1 個 epoch，epoch 1–5 都用它（只測流程）；正式執行 NUM_EPOCHS＝5，照登記
    _rel = 'ckpt/epoch_%d.pth' % min(_ep, NUM_EPOCHS)
    _wsha, _weights = P01_SHA[_rel], 'prereg01/rank/' + _rel
    t1 = time.time()
    if (dump_is_done('inner_' + _name, len(df_val), _syms, _wsha)
            and dump_is_done('val_' + _name, len(parse_val()['feats']), _syms, _wsha)):
        _st = ['skipped', 'skipped']
        print('  %s：inner、本機驗證題都已完成，跳過' % _name)
    else:
        _net, _ = load_net(os.path.join(P01_RANK, _rel), _wsha)
        _st = [dump_inner(_net, _name, _syms, _weights, _wsha), dump_val(_net, _name, _syms, _weights, _wsha)]
        del _net
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
    DUMP_LOG[_name] = {'weights': _weights, 'weights_sha256': _wsha, 'syms': _syms,
                       'inner': _st[0], 'val': _st[1], 'seconds': time.time() - t1}
save_config(dumps_p1=DUMP_LOG)
mark_time('p1_dumps', time.time() - t0)

# ---- 重現核對 ----
t0 = time.time()
V = parse_val()
_parsed = V['keys']['parsed']
with np.load(os.path.join(DUMP_DIR, 'inner_p01e5_d4.npz'), allow_pickle=False) as _z:
    _inner0 = _z['logits'][:, 0, :]
REPRO_INNER_ACC = float((_inner0.argmax(1) == INNER_KEYS['label']).mean() * 100)
with np.load(os.path.join(DUMP_DIR, 'val_p01e5_d4.npz'), allow_pickle=False) as _z:
    _val0 = _z['logits'][:, 0, :]
with open(os.path.join(P01_RANK, 'rank_val_probs.csv'), newline='', encoding='utf-8') as f:
    _p01_rows = list(csv.DictReader(f))

_count_problems = []
if [r['question_id'] for r in _p01_rows] != [q for q, _s, _k in V['spans']]:
    _count_problems.append('題目順序或題數和 prereg01 的 rank_val_probs.csv 不同（%d 對 %d 題）'
                           % (len(V['spans']), len(_p01_rows)))
_max_diff, _b_diff, _a_diff, _compared = 0.0, [], [], 0
if not _count_problems:
    for (_qid, _start, _k), _r in zip(V['spans'], _p01_rows):
        _ok = [_start + j for j in range(_k) if _parsed[_start + j]]
        if int(_r['n_games']) != _k or int(_r['n_parsed']) != len(_ok):
            _count_problems.append('%s：盤數 %d/%d，prereg01 是 %s/%s' % (_qid, len(_ok), _k, _r['n_parsed'], _r['n_games']))
            continue
        if not _ok:
            continue
        # C0：每盤 softmax、同題在 float32 下平均（和 prereg01 推論相同）
        _c0 = [float(v) for v in torch.from_numpy(_val0[_ok]).softmax(1).numpy().mean(0)]
        _old = [float(_r['p_' + lv]) for lv in LEVELS]
        _max_diff = max(_max_diff, max(abs(a - b) for a, b in zip(_c0, _old)))
        _compared += 1
        if rule_b(_c0) != rule_b(_old):
            _b_diff.append(_qid)
        if LEVELS[_argmax_first(_c0)] != LEVELS[_argmax_first(_old)]:
            _a_diff.append(_qid)

GPU_NAME = gpu_name_of(ENV_INFO)   # 這次執行的 GPU（只記錄）
# ② 的 GPU 規則看「寫出 val_p01e5_d4 的那次執行」的 GPU（記在它的完成標記）：斷線後換 GPU 接續，已存的 dump 會跳過，
# 規則不能跟著這次的 GPU 變。正式執行的 dump 一定在 GPU 上算（沒有 GPU 不算），所以沒記到 GPU 就判為不通過
_m_val5 = read_json(_marker_path('val_p01e5_d4')) or {}
DUMP_GPU_NAME = _m_val5.get('gpu_name')
_dump_gpu_ok = SMOKE or DUMP_GPU_NAME is not None
_same_gpu = _dump_gpu_ok and DUMP_GPU_NAME == P01_GPU_NAME
_val_tol = 1e-4 if _same_gpu else 1e-3
# 規則 B 不同的題：正式 ≤ 4 題（1,589 題）；SMOKE 依題數等比例縮小（12 題 → 0 題）
_b_allowed = 4 if not SMOKE else int(4 * len(V['spans']) / 1589)
REPRO = {
    'run': RUN['run'], 'time': now_iso(), 'smoke': SMOKE,
    'gpu_name': GPU_NAME, 'torch': torch.__version__, 'cudnn': ENV_INFO.get('cudnn'),
    'dump_gpu_name': DUMP_GPU_NAME, 'dump_torch': _m_val5.get('torch'), 'dump_cudnn': _m_val5.get('cudnn'),
    'dump_run': _m_val5.get('run'),
    'p01_gpu_name': P01_GPU_NAME, 'p01_torch': P01_TORCH, 'p01_cudnn': P01_CUDNN,
    'same_gpu_as_p01': _same_gpu,
    'same_gpu_rule': '寫出 val_p01e5_d4 的那次執行的 GPU（dumps/_DONE_val_p01e5_d4.json 的 gpu_name）和 prereg01 的 GPU 比',
    'dump_gpu': {'gpu_name': DUMP_GPU_NAME, 'pass': _dump_gpu_ok},
    'inner_acc': {'dump': 'inner_p01e5_d4 s=0', 'value_pct': REPRO_INNER_ACC, 'target_pct': P01_INNER_ACC,
                  'abs_diff_pp': abs(REPRO_INNER_ACC - P01_INNER_ACC), 'tolerance_pp': 0.05,
                  'games': int(len(_inner0)), 'pass': abs(REPRO_INNER_ACC - P01_INNER_ACC) <= 0.05},
    'val_c0_vs_p01': {'dump': 'val_p01e5_d4 s=0', 'reference': 'prereg01/rank/rank_val_probs.csv',
                      'questions_compared': _compared, 'max_abs_diff': _max_diff, 'tolerance': _val_tol,
                      'pass': not _count_problems and _max_diff <= _val_tol},
    'rule_b_diff': {'questions': len(_b_diff), 'allowed': _b_allowed, 'question_ids': _b_diff,
                    'pass': not _count_problems and len(_b_diff) <= _b_allowed},
    'rule_a_diff_info_only': {'questions': len(_a_diff), 'question_ids': _a_diff},
    'counts': {'problems': _count_problems[:20], 'n_problems': len(_count_problems), 'pass': not _count_problems},
}
REPRO['pass'] = all(REPRO[k]['pass'] for k in ('inner_acc', 'val_c0_vs_p01', 'rule_b_diff', 'counts', 'dump_gpu'))
_repro_path = os.path.join(RESULT_DIR, 'repro_check.json')
_prev = read_json(_repro_path)
REPRO['earlier_runs'] = ((_prev.get('earlier_runs') or []) + [{k: v for k, v in _prev.items() if k != 'earlier_runs'}]
                         if _prev else [])
write_json(_repro_path, REPRO)
save_config(repro_check={k: REPRO[k] for k in ('pass', 'gpu_name', 'dump_gpu_name', 'p01_gpu_name', 'same_gpu_as_p01',
                                                 'inner_acc', 'val_c0_vs_p01', 'rule_b_diff', 'counts')})
mark_time('repro_check', time.time() - t0)
print('重現核對：① inner accuracy %.4f%%（prereg01 %.4f%%，差 %.4f 個百分點，容許 0.05）%s'
      % (REPRO_INNER_ACC, P01_INNER_ACC, abs(REPRO_INNER_ACC - P01_INNER_ACC), '通過' if REPRO['inner_acc']['pass'] else '沒過'))
print('          ② 每題平均機率最大差距 %.3g（容許 %g；算 dump 的 GPU %s，prereg01 %s，這次 %s）%s'
      % (_max_diff, _val_tol, DUMP_GPU_NAME, P01_GPU_NAME, GPU_NAME, '通過' if REPRO['val_c0_vs_p01']['pass'] else '沒過'))
print('          ③ 規則 B 不同 %d 題（容許 %d）%s' % (len(_b_diff), _b_allowed, '通過' if REPRO['rule_b_diff']['pass'] else '沒過'))
if _count_problems:
    print('          盤數或題目對不上：', _count_problems[:5])
if not _dump_gpu_ok:
    print('          val_p01e5_d4 的完成標記沒有記到 GPU（正式執行的逐盤輸出必須在 GPU 上算）')
if not REPRO['pass']:
    # 停下前照樣比對 prereg01 指紋、打包（查原因要用逐盤輸出），並把 Drive 的寫入送出
    _after, _p01_changed = record_p01_after()
    RUN.update(end=now_iso(), status='repro_failed')
    save_run_log()
    _zip, _ = pack_results('rank_results_repro_failed.zip')
    if _p01_changed:
        print('另外：prereg01 的目錄在這次執行中被改動（%d 個檔：%s）' % (len(_p01_changed), _p01_changed[:10]))
    flush_drive()
    raise SystemExit('重現核對沒過，依 PREREG_02 停在這裡：不接著訓練、不產生新的 checkpoint。'
                     '處理方式：不要再執行；下載 %s，把這段訊息截圖，一起給 Claude。' % _zip)
print('重現核對都通過')

In [ ]:
# 本版新增：從 prereg01 的 state.pt 接著訓練 EXTRA_EPOCHS 個 epoch（編號 NUM_EPOCHS+1..FINAL_EPOCH）
# 學習率每步線性降到 0；其餘（模型、特徵、batch、fp32、沒有資料增強）和 prereg01 相同。不產生 best_model.pth
# 收尾開關 FINISH_WITHOUT_TRAINING（設定格）打開時不訓練：已完成的 epoch 從 prereg02 的 state.pt 讀回（登記 1.2 的退路）
from torch.optim.lr_scheduler import LambdaLR

P01_STATE_REL = 'ckpt/state.pt'
P01_LAST_REL = 'ckpt/epoch_%d.pth' % NUM_EPOCHS
STATE_PATH = os.path.join(CKPT_DIR, 'state.pt')   # prereg02 自己的 state.pt
T_TOTAL = EXTRA_EPOCHS * P01_STEPS                 # 正式：4 × 2,814 = 11,256 步
if not SMOKE:
    assert P01_STEPS == 2814 and T_TOTAL == 11256, (P01_STEPS, T_TOTAL)
# prereg01 的 state.pt 必須是這個設定（和 prereg01 notebook 的 STATE_CONFIG 相同）
STATE_CONFIG_P01 = {'SEED': SEED, 'BATCH_SIZE': BATCH_SIZE, 'LEARNING_RATE': LEARNING_RATE,
                    'NUM_EPOCHS': NUM_EPOCHS, 'VAL_SIZE': VAL_SIZE, 'features': FEAT_FINGERPRINT}
# prereg02 的 state.pt：設定和這次不同就不接續
STATE_CONFIG_P02 = {'SEED': SEED, 'BATCH_SIZE': BATCH_SIZE, 'PEAK_LR': PEAK_LR,
                    'SCHEDULE': 'LambdaLR: lr = PEAK_LR * max(0, 1 - t / T)；每次 optimizer.step() 後 scheduler.step()',
                    'EXTRA_EPOCHS': EXTRA_EPOCHS, 'T': T_TOTAL, 'P01_STEPS': P01_STEPS, 'NUM_EPOCHS': NUM_EPOCHS,
                    'VAL_SIZE': VAL_SIZE, 'features': FEAT_FINGERPRINT,
                    'init_state_sha256': P01_SHA[P01_STATE_REL], 'init_weights_sha256': P01_SHA[P01_LAST_REL]}
P02_STATE_RESTART = ('在 Drive 把 %s 整個資料夾改名保留（例如 ckpt_broken_日期），再按一次「全部執行」：'
                     '會從 prereg01 的 state.pt 重新接著訓練 %d 個 epoch，逐盤輸出不受影響。把這件事告訴 Claude。'
                     '不要只刪 state.pt，否則舊的 epoch_N.pth 會和新的混在一起。' % (CKPT_DIR, EXTRA_EPOCHS))


def lr_lambda(t):
    return max(0.0, 1.0 - t / T_TOTAL)


def expected_lr(t):
    return PEAK_LR * lr_lambda(t)


def save_state_p02(epoch_done):
    # 這個 epoch 的 epoch_N.pth 存好之後才存 state.pt；存好才算這個 epoch 完成
    tmp = guard_write(STATE_PATH + '.tmp')
    torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
                'global_step': global_step, 'epoch': epoch_done, 'history': history,
                'config': STATE_CONFIG_P02}, tmp)
    os.replace(tmp, guard_write(STATE_PATH))


def _read_checked(path, expect_sha):
    with open(path, 'rb') as f:
        raw = f.read()
    if hashlib.sha256(raw).hexdigest() != expect_sha:
        raise SystemExit('%s 的 SHA-256 和開頭記下的不同（執行中被改動？）。請把這段訊息截圖給 Claude。' % path)
    return raw


def expected_train_steps():
    # 每個 epoch 的步數可以事先算：webdataset 1.0.2 先依 DataLoader worker 在排序後的分片清單上 islice(w, None, workers)
    # 分配分片，再在各 worker 內洗牌；每個 worker 最後一批不滿也算一步（partial=True）。抽特徵的 worker w 依序寫出
    # train-w<w>-000000.tar.gz…，每片 PER_SHARD 盤、最後一片是餘數（_count_w<w>.json 的 written）。回傳 (步數或 None, 說明)
    sizes = {}
    for cpath in sorted(glob.glob(os.path.join(FEAT_DIR, 'train', '_count_w*.json'))):
        wid = int(os.path.basename(cpath)[len('_count_w'):-len('.json')])
        written = int(read_json(cpath)['written'])
        shards = sorted(glob.glob(os.path.join(FEAT_DIR, 'train', 'train-w%d-*.tar.gz' % wid)))
        if len(shards) != -(-written // PER_SHARD):
            return None, ('抽特徵的 worker %d 有 %d 片分片，照 %d 盤、每片 %d 盤應該是 %d 片'
                          % (wid, len(shards), written, PER_SHARD, -(-written // PER_SHARD)))
        for i, s in enumerate(shards):
            sizes[os.path.normpath(s)] = PER_SHARD if i < len(shards) - 1 else written - PER_SHARD * (len(shards) - 1)
    listed = [os.path.normpath(s) for s in train_shards]   # 訓練 loader 實際用的清單和順序
    if sorted(sizes) != sorted(listed):
        return None, '訓練分片清單（%d 片）和 _count_w*.json 推出的分片（%d 片）對不上' % (len(listed), len(sizes))
    if sum(sizes.values()) != train_counts['written']:
        return None, '分片盤數合計 %d，訓練實際寫入 %d' % (sum(sizes.values()), train_counts['written'])
    n_loaders = max(workers, 1)
    per_loader = [sum(sizes[listed[i]] for i in range(w, len(listed), n_loaders)) for w in range(n_loaders)]
    return sum(-(-n // BATCH_SIZE) for n in per_loader if n), {'loader_workers': workers,
                                                               'samples_per_loader_worker': per_loader,
                                                               'shards': len(listed)}


RUN['finish_without_training'] = FINISH_WITHOUT_TRAINING
if FINISH_WITHOUT_TRAINING:
    # 收尾開關：不再訓練（登記 1.2：第 9 個 epoch 沒跑完就只用 prereg01 的輸出）。已完成的 epoch 從 prereg02 的 state.pt 讀回
    t0 = time.time()
    history, global_step, INIT_FROM = [], 0, None
    if os.path.exists(STATE_PATH):
        try:
            _state = torch.load(STATE_PATH, map_location='cpu', weights_only=False)
            history, global_step = list(_state.get('history') or []), int(_state.get('global_step') or 0)
            INIT_FROM = 'prereg02/rank/ckpt/state.pt（只讀回 history）'
            del _state
        except Exception as e:
            print('注意：prereg02 的 ckpt/state.pt 讀不進來（%s: %s），已完成的 epoch 記為沒有' % (type(e).__name__, e))
    save_run_log()
    save_config(training={'finish_without_training': True, 'init_from': INIT_FROM,
                          'epochs_in_state': [h['epoch'] for h in history], 'global_step_in_state': global_step})
    print('收尾開關打開：不接著訓練。prereg02 的 state.pt 記錄已完成的 epoch：%s' % [h['epoch'] for h in history])
    mark_time('train_init', time.time() - t0)
else:
    # 每個 epoch 的步數必須和 prereg01 相同（登記的排程是 EXTRA_EPOCHS × P01_STEPS 步）：訓練前先算好，不同就停下
    _steps, _steps_info = expected_train_steps()
    save_config(train_steps_check={'expected_steps_per_epoch': _steps, 'p01_steps_per_epoch': P01_STEPS,
                                   'detail': _steps_info})
    if _steps != P01_STEPS:
        raise SystemExit('接著訓練每個 epoch 會跑 %s 步，和 prereg01 的 %d 步不同（%s）。登記的學習率排程是 %d × %d 步，'
                         '所以停在這裡，還沒開始訓練（逐盤輸出已存好）。處理方式：不要再執行，把這段訊息截圖給 Claude。'
                         % (_steps, P01_STEPS, _steps_info, EXTRA_EPOCHS, P01_STEPS))
    print('每個 epoch %d 步，和 prereg01 相同（%s）' % (_steps, _steps_info))

    t0 = time.time()
    RUN['loaded_state'] = False
    if os.path.exists(STATE_PATH):
        # 接續：prereg02 自己的 state.pt（自己存的檔，含 history）
        try:
            state = torch.load(STATE_PATH, map_location=DEVICE, weights_only=False)
            _missing = [k for k in ('model', 'optimizer', 'scheduler', 'global_step', 'epoch', 'history', 'config')
                        if k not in state]
        except Exception as e:
            raise SystemExit('prereg02 的 ckpt/state.pt 讀不進來（%s: %s）。處理方式：%s'
                             % (type(e).__name__, e, P02_STATE_RESTART)) from e
        if _missing:
            raise SystemExit('prereg02 的 ckpt/state.pt 缺少 %s。處理方式：%s' % (_missing, P02_STATE_RESTART))
        if state['config'] != STATE_CONFIG_P02:
            _diff = sorted(k for k in set(state['config']) | set(STATE_CONFIG_P02)
                           if state['config'].get(k) != STATE_CONFIG_P02.get(k))
            raise SystemExit('prereg02 的 ckpt/state.pt 的設定和這次不同，不能接續。不同的項目：%s。處理方式：刻意改的話，%s'
                             '不是刻意改的，先查出哪裡變了再跑。' % (_diff, P02_STATE_RESTART))
        try:
            model.load_state_dict(state['model'])
            optimizer.load_state_dict(state['optimizer'])
            scheduler = LambdaLR(optimizer, lr_lambda)
            scheduler.load_state_dict(state['scheduler'])
        except Exception as e:
            raise SystemExit('prereg02 的 ckpt/state.pt 載不進模型、optimizer 或 scheduler（%s: %s）。處理方式：%s'
                             % (type(e).__name__, e, P02_STATE_RESTART)) from e
        start_epoch, global_step, history = state['epoch'], state['global_step'], state['history']
        INIT_FROM = 'prereg02/rank/ckpt/state.pt'
        RUN['loaded_state'] = True
        del state
    else:
        # 第一次：唯讀 prereg01 的 state.pt（第 5 個 epoch 結束，含 AdamW 狀態），核對 SHA-256、設定、權重
        state = torch.load(io.BytesIO(_read_checked(os.path.join(P01_RANK, P01_STATE_REL), P01_SHA[P01_STATE_REL])),
                           map_location=DEVICE, weights_only=False)   # prereg01 自己存的檔，含 history
        _problems = []
        if state.get('epoch') != NUM_EPOCHS:
            _problems.append('epoch 是 %s，應該是 %d' % (state.get('epoch'), NUM_EPOCHS))
        if state.get('config') != STATE_CONFIG_P01:
            _cfg = state.get('config') or {}
            _problems.append('設定不同：%s' % sorted(k for k in set(_cfg) | set(STATE_CONFIG_P01)
                                                    if _cfg.get(k) != STATE_CONFIG_P01.get(k)))
        if not state.get('history') or state['history'][-1].get('steps') != P01_STEPS:
            _problems.append('history 最後一個 epoch 的步數不是 %d' % P01_STEPS)
        if [g['lr'] for g in state['optimizer']['param_groups']] != [PEAK_LR]:
            _problems.append('optimizer 的學習率不是 %g' % PEAK_LR)
        _w_last = torch.load(io.BytesIO(_read_checked(os.path.join(P01_RANK, P01_LAST_REL), P01_SHA[P01_LAST_REL])),
                             map_location='cpu')
        if sorted(_w_last) != sorted(state['model']) or not all(
                torch.equal(state['model'][k].cpu(), _w_last[k]) for k in _w_last):
            _problems.append('state.pt 的權重和 %s 不是逐張量相同' % P01_LAST_REL)
        del _w_last
        if _problems:
            raise SystemExit('prereg01 的 state.pt 和登記的起點不符：%s。%s' % ('；'.join(_problems), P01_HELP))
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        scheduler = LambdaLR(optimizer, lr_lambda)   # 還沒有 initial_lr，會設成現在的 1e-3
        start_epoch, global_step, history = NUM_EPOCHS, 0, []
        INIT_FROM = 'prereg01/rank/ckpt/state.pt'
        del state
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    # LambdaLR 建立時會把學習率設成 λ(0)；接續時照已存的步數重設，確保下一步的學習率正確
    for _g in optimizer.param_groups:
        _g['lr'] = _g['initial_lr'] * lr_lambda(global_step)
    assert scheduler.last_epoch == global_step, (scheduler.last_epoch, global_step)
    assert [g['initial_lr'] for g in optimizer.param_groups] == [PEAK_LR]
    assert abs(optimizer.param_groups[0]['lr'] - expected_lr(global_step)) < 1e-15

    RUN['start_epoch'] = start_epoch
    if start_epoch < FINAL_EPOCH and RUN['loaded_state']:
        run_log['resume_count'] += 1
        RUN.update(resumed_training=True, resume_time=now_iso())
    save_run_log()
    save_config(training={
        'init_from': INIT_FROM, 'start_epoch': start_epoch, 'final_epoch': FINAL_EPOCH, 'global_step_at_start': global_step,
        'state_config': {k: v for k, v in STATE_CONFIG_P02.items() if k != 'features'},
        'optimizer': {'class': type(optimizer).__name__, **optimizer.defaults},
        'criterion': type(criterion).__name__, 'device': str(DEVICE),
        'device_name': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else None,
        'tf32': {'cudnn': torch.backends.cudnn.allow_tf32, 'matmul': torch.backends.cuda.matmul.allow_tf32},
        'loader_workers': workers, 'seed_rule': 'random／numpy／torch 的 seed 在每個 epoch 開始時設為 SEED + epoch',
        'checkpoint_rule': '指定 epoch_%d.pth（最後一個 epoch），不另選 best；inner 驗證只當監看' % FINAL_EPOCH})
    save_config_summary(resumed=run_log['resume_count'] > 0, resume_count=run_log['resume_count'])
    if start_epoch >= FINAL_EPOCH:
        print('epoch %d–%d 都已訓練完（global_step %d），這次不訓練' % (NUM_EPOCHS + 1, FINAL_EPOCH, global_step))
    else:
        print('從 %s 開始：epoch %d → %d，global_step %d／%d，學習率 %.3g'
              % (INIT_FROM, start_epoch + 1, FINAL_EPOCH, global_step, T_TOTAL, optimizer.param_groups[0]['lr']))
    mark_time('train_init', time.time() - t0)

    model.train()
    for epoch in range(start_epoch, FINAL_EPOCH):
        t0 = time.time()
        random.seed(SEED + epoch + 1)
        np.random.seed(SEED + epoch + 1)
        torch.manual_seed(SEED + epoch + 1)
        total_loss, correct, total, steps = 0.0, 0, 0, 0
        lr_start = optimizer.param_groups[0]['lr']

        for inputs, labels, _ in train_loader:
            inputs = inputs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            scheduler.step()      # 本版：每步更新學習率
            global_step += 1

            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            steps += 1
            if steps % 500 == 0:
                print('  ep%d step %d  loss %.4f  acc %.2f%%  lr %.3g  %.1f 分'
                      % (epoch + 1, steps, total_loss / steps, 100 * correct / total,
                         optimizer.param_groups[0]['lr'], (time.time() - t0) / 60), flush=True)

        assert abs(optimizer.param_groups[0]['lr'] - expected_lr(global_step)) < 1e-12, 'learning-rate schedule drifted'
        train_acc = 100 * correct / total if total else 0.0
        val_loss, val_acc = validate(model, val_loader, criterion)   # inner 逐盤 accuracy：只當監看
        history.append({'epoch': epoch + 1, 'train_acc': train_acc, 'val_acc': val_acc,
                        'train_loss': total_loss / steps if steps else 0.0, 'val_loss': val_loss,
                        'steps': steps, 'samples': total, 'seconds': time.time() - t0, 'run': RUN['run'],
                        'lr_start': lr_start, 'lr_end': optimizer.param_groups[0]['lr'], 'global_step': global_step})
        print('Epoch %d/%d  train acc %.2f%%  |  inner loss %.4f acc %.2f%%  |  lr %.3g → %.3g  |  %.1f 分'
              % (epoch + 1, FINAL_EPOCH, train_acc, val_loss, val_acc, lr_start, optimizer.param_groups[0]['lr'],
                 (time.time() - t0) / 60), flush=True)

        _tmp = guard_write(os.path.join(CKPT_DIR, 'epoch_%d.pth.tmp' % (epoch + 1)))
        torch.save(model.state_dict(), _tmp)
        os.replace(_tmp, guard_write(os.path.join(CKPT_DIR, 'epoch_%d.pth' % (epoch + 1))))
        save_state_p02(epoch + 1)   # 最後才存；存好才算這個 epoch 完成
        pd.DataFrame(history).to_csv(guard_write(os.path.join(CKPT_DIR, 'history.csv')), index=False)
        RUN['epochs_trained'].append(epoch + 1)
        save_run_log()
        save_config(epochs_trained=list(RUN['epochs_trained']))
        save_config_summary(epochs_completed=epoch + 1, global_step=global_step)
        mark_time('epoch_%d' % (epoch + 1), time.time() - t0)

if history:
    pd.DataFrame(history).to_csv(guard_write(os.path.join(CKPT_DIR, 'history.csv')), index=False)
TRAIN_COMPLETE = ([h['epoch'] for h in history] == list(range(NUM_EPOCHS + 1, FINAL_EPOCH + 1))
                  and global_step == sum(h['steps'] for h in history))
save_config_summary(training_complete=TRAIN_COMPLETE, epochs_completed=history[-1]['epoch'] if history else NUM_EPOCHS,
                    trained_in_runs=sorted({h['run'] for h in history}), finish_without_training=FINISH_WITHOUT_TRAINING)
print('接著訓練%s：epoch %s，global_step %d%s'
      % ('完成' if TRAIN_COMPLETE else '沒有完成', [h['epoch'] for h in history], global_step,
         '（收尾開關打開，這次不訓練）' if FINISH_WITHOUT_TRAINING
         else '，最後的學習率 %.3g' % optimizer.param_groups[0]['lr']))

In [ ]:
# 本版新增：P2 逐盤輸出（接著訓練的最後一個 epoch，正式是 epoch_9.pth，8 種對稱）
# 第 FINAL_EPOCH 個 epoch 沒完成就跳過並記錄（本機會照登記拿掉用到它的組合）
t0 = time.time()
P2_NAME = 'p02e9_d4'
_p2_path = os.path.join(CKPT_DIR, 'epoch_%d.pth' % FINAL_EPOCH)
if TRAIN_COMPLETE and os.path.exists(_p2_path):
    _net, _wsha = load_net(_p2_path)
    _weights = 'prereg02/rank/ckpt/epoch_%d.pth' % FINAL_EPOCH
    _st = [dump_inner(_net, P2_NAME, NUM_SYMS, _weights, _wsha), dump_val(_net, P2_NAME, NUM_SYMS, _weights, _wsha)]
    del _net
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    with np.load(os.path.join(DUMP_DIR, 'inner_%s.npz' % P2_NAME), allow_pickle=False) as _z:
        _acc9 = float((_z['logits'][:, 0, :].argmax(1) == INNER_KEYS['label']).mean() * 100)
    P2_RESULT = {'done': True, 'weights': _weights, 'weights_sha256': _wsha, 'inner': _st[0], 'val': _st[1],
                 'inner_acc_from_dump_pct': _acc9, 'inner_acc_from_history_pct': history[-1]['val_acc'],
                 'abs_diff_pp': abs(_acc9 - history[-1]['val_acc'])}
    if P2_RESULT['abs_diff_pp'] > 0.05:
        print('注意：epoch %d 的 inner accuracy，dump 算出 %.4f%%，訓練時的監看是 %.4f%%，差超過 0.05 個百分點。'
              '結果照樣打包，回報 Claude 時提一下。' % (FINAL_EPOCH, _acc9, history[-1]['val_acc']))
    print('P2 逐盤輸出完成：%s（epoch %d 的 inner accuracy %.4f%%）' % (P2_NAME, FINAL_EPOCH, _acc9))
else:
    P2_RESULT = {'done': False, 'reason': '第 %d 個 epoch 沒有完成（TRAIN_COMPLETE=%s，%s %s%s）'
                 % (FINAL_EPOCH, TRAIN_COMPLETE, _p2_path, '存在' if os.path.exists(_p2_path) else '不存在',
                    '；收尾開關打開' if FINISH_WITHOUT_TRAINING else '')}
    print('跳過 P2 逐盤輸出：%s' % P2_RESULT['reason'])
P2_RESULT['finish_without_training'] = FINISH_WITHOUT_TRAINING
save_config(dumps_p2=P2_RESULT)
mark_time('p2_dumps', time.time() - t0)

In [ ]:
# 本版改寫的結尾：核對所有逐盤輸出，比對 prereg01 目錄前後的指紋，補完 config／timings／run_log，
# 打包 rank_results.zip（不含權重），把 Drive 的寫入送出
import zipfile

t0 = time.time()
V = parse_val()
_parsed = V['keys']['parsed']
_expect = ['inner_keys', 'val_keys'] + ['%s_%s' % (p, n) for n, _e, _s in P1_DUMPS for p in ('inner', 'val')]
if P2_RESULT['done']:
    _expect += ['inner_' + P2_NAME, 'val_' + P2_NAME]
DUMP_CHECK = {}
for stem in _expect:
    _path, _m = _dump_path(stem), read_json(_marker_path(stem))
    _sha = sha256_path(_path) if os.path.exists(_path) else None
    assert _m is not None and _sha == _m['sha256'], ('完成標記不符', stem)
    _rows_expected = len(df_val) if stem.startswith('inner') else len(V['feats'])
    with np.load(_path, allow_pickle=False) as _z:
        if stem.endswith('_keys'):
            _rows = len(_z['game_id'] if stem.startswith('inner') else _z['question_id'])
            assert all(len(_z[k]) == _rows for k in _z.files), stem
        else:
            _L = _z['logits']
            _rows = _L.shape[0]
            assert _L.dtype == np.float32 and _L.ndim == 3 and _L.shape[1:] == (int(_z['syms']), NUM_CLASSES), stem
            assert str(_z['weights_sha256']) == _m['weights_sha256'], stem
            if stem.startswith('val_'):
                assert np.array_equal(np.isnan(_L).all(axis=(1, 2)), ~_parsed) and not np.isnan(_L[_parsed]).any(), stem
            else:
                assert not np.isnan(_L).any(), stem
    assert _rows == _rows_expected == _m['rows'], (stem, _rows, _rows_expected, _m['rows'])
    assert os.path.getsize(_path) < DUMP_MAX_BYTES, stem
    DUMP_CHECK[stem] = {'rows': _rows, 'bytes': os.path.getsize(_path), 'sha256': _sha,
                        'weights': _m['weights'], 'weights_sha256': _m['weights_sha256']}

# prereg01 目錄的指紋（結尾）：必須和這次開頭記的完全相同
P01_AFTER, _p01_changed = record_p01_after()

OUTPUTS = {
    'dumps': DUMP_CHECK,
    'inner_games': len(df_val), 'val_questions': len(V['spans']), 'val_games': len(V['feats']),
    'val_games_unparsed': int((~_parsed).sum()),
    'repro_check_pass': REPRO['pass'], 'training_complete': TRAIN_COMPLETE, 'p2_dumps_done': P2_RESULT['done'],
    'finish_without_training': FINISH_WITHOUT_TRAINING,
    'p01_readonly_unchanged': not _p01_changed, 'p01_files_fingerprinted': len(P01_AFTER),
    'history': history,
    'files': {n: os.path.getsize(os.path.join(RESULT_DIR, n)) for n in sorted(os.listdir(RESULT_DIR))
              if os.path.isfile(os.path.join(RESULT_DIR, n))},
}
RUN.update(end=now_iso(), status='completed' if not _p01_changed else 'p01_changed')
save_config(outputs=OUTPUTS, end=RUN['end'])
save_run_log()
mark_time('final_check', time.time() - t0)
mark_time('total_this_run', time.time() - RUN_START)
print(json.dumps({k: v for k, v in OUTPUTS.items() if k not in ('dumps', 'history')}, ensure_ascii=False, indent=1))
print('ckpt/', sorted(os.listdir(CKPT_DIR)))

# 結果打包：結果目錄裡除了權重和暫存檔以外的檔案，打成一個 zip 放在結果目錄旁邊
RESULTS_ZIP, _skipped_big = pack_results(os.path.basename(RESULT_DIR) + '_results.zip')
with zipfile.ZipFile(RESULTS_ZIP) as _zf:
    _names = set(_zf.namelist())
_missing_in_zip = [s for s in _expect if 'rank/dumps/%s.npz' % s not in _names]
assert not _missing_in_zip, ('dump 沒進 zip', _missing_in_zip)
_zip_mb = os.path.getsize(RESULTS_ZIP) / 1e6
# 下載 zip、刪除執行階段之前，先把 Drive 的寫入全部送出（epoch_9.pth、state.pt 之後還要用）
flush_drive()
if _p01_changed:
    raise SystemExit('prereg01 的目錄在這次執行中被改動（%d 個檔：%s）。結果已打包在 %s，但先不要用；'
                     '把這段訊息截圖給 Claude。' % (len(_p01_changed), _p01_changed[:10], RESULTS_ZIP))
print('完成。請下載這個檔回傳給 Claude（合併、校準、打分都在本機做）：%s（%.1f MB）' % (RESULTS_ZIP, _zip_mb))